### **Student Information**
GitHub ID: ken931014-ctrl

Kaggle name:Ken_pomelo (username: leeyukuan10)

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

# Project Report

## 1. Model Development

### 1.1 Preprocessing Steps

Lab2主要對一個英文貼文分成 anger、disgust、fear、joy、sadness、surprise 六種情緒。資料共 4,800 筆，其中train data 為3,360 筆、test data 為 1,440 筆。我先用 `post_id` 合併貼文、官方 train/test 分組與情緒標籤，使用 one-to-one merge 逐一檢查對應關係，確認訓練標籤完整、測試標籤未提供，再去核對 sample submission 的 ID。

在 clean過程中包含 HTML entity 還原、Unicode NFKC 正規化、轉小寫、URL 與使用者名稱替換，以及合併多餘空白。清理結果 `clean` 用於 TF-IDF 與分組；Transformer 則輸入原始 `text`，保留大小寫與標點。兩者的用途不同，並非所有模型都使用清理後文字。

驗證我則是用 `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` 的方式來完成，以 `clean` 作為 group，讓相同清除文字不去跨越訓練與驗證集，同時我也盡量保留類別比例。TF-IDF 詞彙表與 IDF 僅由各次訓練資料估計。分組只能隔開清理後完全相同的文字，所以我認為無法排除所有近似貼文或公開預訓練資料的潛在重疊。

### 1.2 Feature Engineering Steps

一開始使用 word TF-IDF unigram，之後加入 word bigram 及 `char_wb` 字元 3–5 gram。字元特徵上限為 50,000，目的是補充拼字變化與局部字串資訊。以 FeatureUnion 結合兩種Feature，再輸入 LinearSVC；比較 C、class_weight 與 min_df。第一組驗證中，word TF-IDF + SVM 為 0.4712，word + character TF-IDF + SVM 為 0.5024。GridSearchCV 的最佳分數屬於用來選參數的交叉驗證結果，這裡我並沒有視為獨立測試成績。

Transformer 則使用各模型自己的 tokenizer 與學得的上下文表示，這裡則沒有額外去做拼接 TF-IDF。主要微調實驗截斷至 128 tokens，採用 dynamic padding。Hartmann (anger 🤬disgust 🤢fear 😨joy 😀neutral 😐sadness 😭surprise 😲，取自於"https://huggingface.co/j-hartmann/emotion-english-roberta-large" )情緒模型原有七類，保留其分類頭訓練，推論時依 config 的標籤名稱取出競賽六類 logits，再做 softmax。

### 1.3 Explanation of Your Model

我先建立 LinearSVC baseline，再嘗試 Hugging Face 情緒模型直接推論、DistilRoBERTa 微調，以及 DeBERTa-v3-base 微調。

- 因為在`j-hartmann/emotion-english-distilroberta-base` 中已經做過英文情緒分類微調。所以在lab2中我只再對競賽訓練資料微調。主要的原版設定為 AdamW、learning rate 5e-6、weight decay 0.01、batch size 8、gradient clipping 1.0。Fold 1 曾觀察到第三個 epoch，最佳保存於 epoch 2；Fold 2~5 則是訓練兩個 epoch並保存最佳驗證 checkpoint。
- `microsoft/deberta-v3-base` 是通用pretrain model，為lab2建立六類分類。設定為 AdamW、learning rate 2e-5、weight decay 0.01、batch size 4、gradient accumulation 4、FP16 autocast 與 GradScaler，搭配 10% warmup 的線性 scheduler，我設定最多四個 epoch並保存最佳 checkpoint。
- 另外我還有特別比較了類別加權 loss、全資料訓練、seed 42/123 (自訂) 融合、模型融合比例和幾何平均與溫度調整。加權 loss 後的效果並未優於原版，因此我最後沒有用於最終5 Fold 50/50 的模型中。

5 fold版本分別訓練五個 DistilRoBERTa 與5個 DeBERTa。對train set先統一六類標籤順序，每種模型取5 fold機率平均，然後再將兩種架構各占 50% 去做平均，以最大機率類別去作為預測。最後依照 sample submission 的 ID 去做最後的排序，輸出 `id,label`，檢查筆數、ID 唯一性和缺值與標籤集合。

OOF 評估中，每筆訓練貼文只使用未以該筆訓練的 fold checkpoint 預測，在這裡不使用其餘四折模型一起評估它。但是驗證集也用來挑選 epoch與融合設定，因此 OOF 仍有模型選擇的樂觀偏差；不是完全獨立的測試。後期的四折選溫度、另一折檢查亦不是完整 nested CV，因各模型訓練資料與 checkpoint 選擇並未在外層重新隔離。

note:「溫度」:溫度參數(temperature parameter(T))用來調整模型預測機率的集中程度

### 實驗結果與最終選取

| 方法 | Public | Private | 截止前選取 |
|---|---:|---:|---|
| 單組分割 DistilRoBERTa + DeBERTa，50/50 | 0.6236 | 0.6308 | 否 |
| 全訓練資料 DistilRoBERTa + DeBERTa，50/50 | 0.6246 | 0.6160 | 是 |
| 全資料 seed 42/123 融合，50/50 | 0.6200 | 0.6086 | 否 |
| 五折 DistilRoBERTa + DeBERTa，50/50 | 0.6458 | 0.6095 | 是 |
| 五折 DistilRoBERTa + DeBERTa，65/35 | 0.6207 | 0.6186 | 否 |
| 五折 50/50，Distil T=1、DeBERTa T=1.5 | 0.6244 | 0.6124 | 否 |

截止前選取五折 50/50 與全資料 50/50。正式 Private 成績為兩份選取中的較高值 **0.6160，Private 排名第 72 名**

(居然是掉最多的排名，掉了整整57名 (つД`)ノ ，Man!, What can I say , score out~ ToT...)



### 結論與限制

本次從 TF-IDF baseline 逐步進展至pretrain model微調與五折融合，完成模型保存、機率對齊、OOF 評估與提交檢查，但最後選取的兩個版本未包含事後 Private 分數最高的版本。五折 50/50 融合的 Public 分數為 0.6458，Private 分數為 0.6095，相差 0.0363，顯示 Public 上的改善沒有在 Private 上得到相同結果。

我認為之所以這次public and private 的分數差距是因為反覆參考 Public 分數調整設定與選擇提交版本，可能造成對 Public leaderboard 的選擇因而導致overfitting。實驗過程中比較了多種模型、融合比例與溫度設定，若持續根據同一個 Public 子集的結果決定下一步，便可能選到剛好適合該public的子集、卻未必能泛化到其他資料的版本。

至於pretrain model與5fold訓練，不能直接認定為overfitting的原因。pretrain model在有限資料上微調確實存在overfitting風險，但我認為仍需透過訓練與驗證曲線確認；5fold融合則通常有助於降低資料切分造成的變異，無法因為這次 Private 較低，就判定五折使模型overfitting(雖然很傷心哈哈)。測試子集的難度、類別分布與資料量，也都可能影響分數，目前的結果不足以區分各因素的影響。

如果能rerun lab2，我會在實驗開始前固定驗證方式與最終模型選取規則，主要依據本地驗證及 OOF 結果做決策，減少依 Public 分數反覆調整。同時保留未參與調參的獨立驗證集；再使用 nested cross-validation 評估整個選模流程，並檢查各類別 precision、recall、F1 與易混淆的案例。Pretrain model是否與本次資料重疊也需要另檢查。

### 1.4 AI Prompts, what and why you decide to use that technique suggested by AI

lab2 我是使用 ChatGPT 協助解釋模型、撰寫與除錯程式、設計實驗、比較結果，以及整理本報告。
訓練則是在本機 Jupyter 執行。
以下依實驗流程整理主要提問、AI 協助內容，以及實際採用與結果。部分實驗方向是在與 AI 討論後決定嘗試。

| 提問摘要 | AI 協助內容 | 採用與結果 |
|---|---|---|
| 幫我去 Hugging Face 搜尋，有沒有適合我們這次 Kaggle 情緒分類的 pretrained model。 | 搜尋並比較適合情緒分類的pretrain model。 | 嘗試 Hartmann 系列模型，後續加入 DeBERTa。 |
| 先直接用 pretrained model 預測看看，我需要怎麼使用？ | 協助 tokenizer、模型載入、類別對應及提交 CSV 產生。 | 執行直接推論，作為後續微調的比較基準。 |
| 現在顯示 CUDA available 是 False，幫我看為什麼沒有用到 GPU。 | 檢查 PyTorch 與 CUDA 環境。 | 修復環境後使用 GPU 進行訓練。 |
| 那先用 DistilRoBERTa 微調，看看能不能比直接預測更好。 | 建立訓練與驗證流程，保存最佳 checkpoint。 | 實際執行微調，依 validation macro F1 選擇 checkpoint。 |
| 幫我比較不同 learning rate 和 epoch，看看哪個設定比較好。 | 調整訓練參數，比較各輪驗證表現。 | 依驗證結果選擇設定，避免只採用最後一輪模型。 |
| 不同設定的 DistilRoBERTa 可以融合嗎？幫我比較看看。 | 對齊類別順序，平均不同模型的預測機率。 | 比較融合與單模型的驗證結果。 |
| 接下來試 DeBERTa，看看和 DistilRoBERTa 比起來如何。 | 協助模型載入、分類頭設定與微調。 | 完成 DeBERTa 訓練，並加入後續融合實驗。 |
| 幫我做兩個模型 50/50、25/75、75/25 的融合版本。 | 實作機率加權平均，比較 validation macro F1。 | 比較不同融合比例，作為後續實驗的依據。 |
| 接下來用全部訓練資料重新訓練，再做兩個模型 50/50 融合。 | 建立 full-data training 與融合推論流程。 | 全資料 50/50 取得 Public 0.6246，後續選為最終提交之一。 |
| 也產生 DeBERTa 單模型的提交檔，我想和融合結果比較。 | 產生全資料 DeBERTa 的測試預測與提交檔。 | Public 0.5919，未超越兩模型融合。 |
| 再試不同 random seed，把預測平均，看看有沒有比較穩定。 | 協助不同種子訓練與預測機率平均。 | 多種子融合 Public 0.6200，未超越原本的全資料融合。 |
| 我想試類別加權 loss，看看能不能改善 macro F1。 | 根據訓練類別數量設定 loss 權重，建立比較流程。 | Weighted 50/50 驗證分數 0.5937，低於原始版本的 0.6098。 |
| Weighted 50/50 不行，那再試 25/75 和 75/25 呢？ | 比較加權模型的不同融合比例。 | 測試後仍未超越原始未加權版本，未用於最終主要模型。 |
| 接下來做五折交叉驗證，兩個模型都跑完，並保存每折結果。 | 建立分組切分、各折訓練、checkpoint 與預測保存流程。 | 完成 DistilRoBERTa 與 DeBERTa 的五折訓練。 |
| 機率檔讀取出錯，而且兩個模型的機率完全一樣，幫我檢查原因。 | 檢查 npz 讀取與推論流程，從各自 checkpoint 重新推論。 | 修復比較流程，確認兩模型預測不同後再進行融合。 |
| 把各自五折的測試機率平均，再做兩個模型 50/50 融合。 | 實作跨折平均與兩模型融合，產生提交檔。 | Public 0.6458，為本次實驗最高 Public 分數。 |
| 用完整 OOF 比較 50/50、65/35、75/25，也列出單模型結果。 | 計算不同融合比例的 OOF macro F1。 | 65/35 的 OOF 較高，但提交 Public 0.6207，未改善。 |
| 除了算術平均，也試試幾何平均，看看融合方式有沒有影響。 | 實作幾何平均，使用相同 OOF 資料比較。 | 幾何平均 OOF 0.621680，低於算術平均的 0.622207。 |
| 再試 temperature 調整，看看改變機率分布後會不會比較好。 | 比較溫度組合，並以其他四折選設定、剩餘一折檢查。 | 檢查不同折的表現，作為溫度設定的選擇依據。 |
| 幫我產生 DistilRoBERTa T=1、DeBERTa T=1.5 的提交版本。 | 調整模型機率後融合，產生提交檔。 | Public 0.6244，未超越原始五折 50/50。 |
| 最後幫我整理全部實驗，看看應該選哪兩份提交，也檢討 Public 和 Private 的差異。 | 整理驗證、OOF 與提交結果，協助比較最終版本。 | 選擇五折 50/50 與全資料 50/50；最終官方 Private 分數為 0.6160。 |
AI 提供了大部分實驗程式骨架與除錯建議，並影響了最後模型選取。我沒有將所有建議都視為有效：類別加權、65/35 與溫度調整均依實測記錄其限制。

### References

1. Hartmann, J. (2022). [Emotion English DistilRoBERTa-base](https://huggingface.co/j-hartmann/emotion-english-distilroberta-base).
2. [Hartmann Emotion English RoBERTa-large](https://huggingface.co/j-hartmann/emotion-english-roberta-large).
3. Microsoft. [DeBERTa-v3-base model card](https://huggingface.co/microsoft/deberta-v3-base).
4. He, P., Gao, J., & Chen, W. (2021). [DeBERTaV3](https://arxiv.org/abs/2111.09543).


**`From here on starts the code section for the competition.`**

---

## 執行與重現說明（提交前請閱讀）

本整理版新增報告、修復 Fold 1 原始程式與輸出不一致、移除自動重裝環境和過早模型登記，並將模型登記改成相對路徑。修改後的訓練格尚未在此環境實際重跑；不能宣稱已通過 Restart & Run All，亦不能保證新訓練逐位重現既有 Kaggle CSV。原始 Notebook 另放於壓縮包的 `archive/`，保留競賽時的輸出與除錯過程。

需要競賽四個原始檔置於 `data/competition/`、可用 CUDA GPU，以及 torch、transformers、sentencepiece、protobuf、scikit-learn、pandas、numpy、tqdm。先在課程環境安裝相容依賴，再執行；不要把強制重裝 torch 當作日常步驟。五折模型訓練需時間與磁碟空間。

重建 Fold 1 的兩個 learning rate 實驗屬於修復程式路徑，不是競賽結束後的新提交。現有 checkpoint 請先備份：程式中的固定 Fold 1 保存目錄在重跑時會更新。其餘 fold 使用時間戳目錄。GPU 非確定性、套件版本與 HF 權重 revision 未完全鎖定，均限制精確重現。

模型訓練資料與 `.env`、金鑰不應混入公開附件。這份 ZIP 只是 Homework 整理包，不是完整 Lab 2 繳交 ZIP；另需 Master、AgentPipeline 與兩份原始加密 logs。`ai_prompts.md` 完整提示紀錄仍須由學生補齊。


# Competition Code

## 執行導覽

本 Notebook 同時保留最終提交流程及報告討論的補充實驗。為避免移動 cell 造成全域變數依賴失效，實驗仍依發展順序排列，使用標題區別用途。不要只刪掉模型定義或前置資料 cell。

| 區塊 | 目的 |
|---|---|
| Preprocessing / Feature Engineering / A1–A4 | baseline、特徵、早期模型比較 |
| Fold 1 原版訓練、DeBERTa、B1 | 建立第一折與早期融合實驗 |
| 最終提交二：全資料 seed 42 | 全資料 50/50，正式選取，Private 0.6160 |
| Fold 2–5、最終提交一 | 五折 50/50，正式選取，Private 0.6095 |
| 補充實驗與分析 | 多 seed、加權、比例、幾何平均、溫度調整 |

已刪除自動安裝／強制重裝套件、重複硬體檢查、無作用的舊診斷與過早模型登記。完整歷史紀錄保留於原始 Notebook。

依賴請於執行前另行安裝：numpy、pandas、scikit-learn、torch（相容 CUDA 版本）、transformers、sentencepiece、protobuf、tqdm。此整理版未重跑 GPU 訓練，不保證既有 checkpoint 的逐位重現。

## 1. Preprocessing Steps


In [2]:
### Add the code related to the preprocessing steps in cells inside this section
#讀取資料。
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data/competition")
SEED = 42

posts = pd.read_json(DATA_DIR / "final_posts.json")
split_info = pd.read_csv(
    DATA_DIR / "data_identification.csv", dtype=str
)
labels = pd.read_csv(DATA_DIR / "emotion.csv", dtype=str)
sample_submission = pd.read_csv(
    DATA_DIR / "sampleSubmission.csv", dtype=str
)

data = split_info.merge(
    posts,
    on="post_id",
    how="left",
    validate="one_to_one",
)
data = data.merge(
    labels,
    on="post_id",
    how="left",
    validate="one_to_one",
)

train = data.loc[data["set"] == "train"].copy()
test = data.loc[data["set"] == "test"].copy()

assert train["text"].notna().all()
assert test["text"].notna().all()
assert train["label"].notna().all()
assert test["label"].isna().all()
assert set(test["post_id"]) == set(sample_submission["post_id"])

print("Training rows:", len(train))
print("Test rows:", len(test))
print("\nTraining label counts:")
print(train["label"].value_counts())

display(train[["post_id", "text", "label"]].head())

Training rows: 3360
Test rows: 1440

Training label counts:
label
joy         875
anger       674
sadness     628
disgust     454
surprise    441
fear        288
Name: count, dtype: int64


,post_id,text,label
0,f8bv5zh,Thought it looked weird as well.,disgust
1,q0rr1l9,Bear died in a pit,disgust
2,p94rwgk,I have endured worse.,sadness
3,9u6wtvm,The pottery around Brasov is in danger,fear
4,h8ea329,"Meanwhile, our soldiers were approaching the a...",fear


In [3]:
#清理文字
import re
import html
import unicodedata

def clean_text(text):
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()

    
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)

    
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["clean"] = train["text"].map(clean_text)
test["clean"] = test["text"].map(clean_text)

display(train[["text", "clean", "label"]].head())

,text,clean,label
0,Thought it looked weird as well.,thought it looked weird as well.,disgust
1,Bear died in a pit,bear died in a pit,disgust
2,I have endured worse.,i have endured worse.,sadness
3,The pottery around Brasov is in danger,the pottery around brasov is in danger,fear
4,"Meanwhile, our soldiers were approaching the a...","meanwhile, our soldiers were approaching the a...",fear


In [4]:
#建立 train_part、`valid_part`
from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

train_idx, valid_idx = next(
    splitter.split(
        X=train["clean"],
        y=train["label"],
        groups=train["clean"],
    )
)

train_part = train.iloc[train_idx].copy()
valid_part = train.iloc[valid_idx].copy()


assert set(train_part["clean"]).isdisjoint(
    set(valid_part["clean"])
)

print("用於訓練：", len(train_part))
print("用於驗證：", len(valid_part))

display(
    pd.DataFrame({
        "train": train_part["label"].value_counts(),
        "validation": valid_part["label"].value_counts(),
    }).fillna(0).astype(int)
)

用於訓練： 2687
用於驗證： 673


,train,validation
label,,
joy,700,175
anger,539,135
sadness,502,126
disgust,363,91
surprise,353,88
fear,230,58


## 2. Feature Engineering Steps

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=2,
)


X_train = vectorizer.fit_transform(train_part["clean"])
X_valid = vectorizer.transform(valid_part["clean"])

y_train = train_part["label"]
y_valid = valid_part["label"]

print("訓練特徵：", X_train.shape)
print("驗證特徵：", X_valid.shape)### Add the code related to the feature engineering steps in cells inside this section

訓練特徵： (2687, 2518)
驗證特徵： (673, 2518)


## 3. Model Implementation Steps

In [6]:
%pip install --upgrade torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### A1. LinearSVC baseline 與錯誤分析 (LinearSVC Baseline and Error Analysis)
 

In [7]:
### Add the code related to the model implementation steps in cells inside this section
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)

baseline_model = LinearSVC(
    C=1.0,
    random_state=SEED,
)

baseline_model.fit(X_train, y_train)
valid_pred = baseline_model.predict(X_valid)

baseline_f1 = f1_score(
    y_valid, valid_pred, average="macro"
)

print(f"Validation accuracy: {accuracy_score(y_valid, valid_pred):.4f}")
print(f"Validation macro F1: {baseline_f1:.4f}")
print()
print(classification_report(
    y_valid,
    valid_pred,
    digits=4,
    zero_division=0,
))

Validation accuracy: 0.4695
Validation macro F1: 0.4479

              precision    recall  f1-score   support

       anger     0.3986    0.4370    0.4170       135
     disgust     0.3544    0.3077    0.3294        91
        fear     0.5833    0.3621    0.4468        58
         joy     0.5561    0.6229    0.5876       175
     sadness     0.4961    0.5079    0.5020       126
    surprise     0.4118    0.3977    0.4046        88

    accuracy                         0.4695       673
   macro avg     0.4667    0.4392    0.4479       673
weighted avg     0.4695    0.4695    0.4664       673



In [8]:
from sklearn.pipeline import Pipeline

experiment_rows = [{
    "features": "unigram",
    "C": 1.0,
    "class_weight": None,
    "macro_f1": baseline_f1,
}]

candidate_models = {}

for c in [0.25, 1.0, 4.0]:
    for weight in [None, "balanced"]:
        model = Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
            )),
            ("svm", LinearSVC(
                C=c,
                class_weight=weight,
                random_state=SEED,
            )),
        ])

        model.fit(train_part["clean"], y_train)
        predictions = model.predict(valid_part["clean"])

        score = f1_score(
            y_valid, predictions, average="macro"
        )

        key = (c, weight)
        candidate_models[key] = model

        experiment_rows.append({
            "features": "unigram + bigram",
            "C": c,
            "class_weight": weight,
            "macro_f1": score,
        })

experiment_results = pd.DataFrame(experiment_rows).sort_values(
    "macro_f1", ascending=False
)

display(experiment_results.round(4))

,features,C,class_weight,macro_f1
2,unigram + bigram,0.25,balanced,0.4712
4,unigram + bigram,1.00,balanced,0.4510
0,unigram,1.00,NaN,0.4479
1,unigram + bigram,0.25,NaN,0.4458
3,unigram + bigram,1.00,NaN,0.4406
6,unigram + bigram,4.00,balanced,0.4190
5,unigram + bigram,4.00,NaN,0.4139


In [9]:
best_model = candidate_models[(0.25, "balanced")]

best_valid_pred = best_model.predict(valid_part["clean"])

print(classification_report(
    y_valid,
    best_valid_pred,
    digits=4,
    zero_division=0,
))

              precision    recall  f1-score   support

       anger     0.4406    0.4667    0.4532       135
     disgust     0.3678    0.3516    0.3596        91
        fear     0.5102    0.4310    0.4673        58
         joy     0.6105    0.6000    0.6052       175
     sadness     0.4766    0.4841    0.4803       126
    surprise     0.4468    0.4773    0.4615        88

    accuracy                         0.4874       673
   macro avg     0.4754    0.4685    0.4712       673
weighted avg     0.4885    0.4874    0.4874       673



In [10]:
from sklearn.metrics import confusion_matrix

label_order = sorted(train["label"].unique())

cm = confusion_matrix(
    y_valid,
    best_valid_pred,
    labels=label_order,
)

display(pd.DataFrame(
    cm,
    index=[f"True: {label}" for label in label_order],
    columns=[f"Pred: {label}" for label in label_order],
))

,Pred: anger,Pred: disgust,Pred: fear,Pred: joy,Pred: sadness,Pred: surprise
True: anger,63,19,5,15,18,15
True: disgust,22,32,9,11,11,6
True: fear,5,6,25,8,11,3
True: joy,16,13,4,105,20,17
True: sadness,22,8,4,20,61,11
True: surprise,15,9,2,13,7,42


In [11]:
unigram_balanced = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 1),
        min_df=2,
    )),
    ("svm", LinearSVC(
        C=0.25,
        class_weight="balanced",
        random_state=SEED,
    )),
])

unigram_balanced.fit(train_part["clean"], y_train)

unigram_balanced_pred = unigram_balanced.predict(
    valid_part["clean"]
)

unigram_balanced_f1 = f1_score(
    y_valid,
    unigram_balanced_pred,
    average="macro",
)

print(f"Unigram + balanced: {unigram_balanced_f1:.4f}")
print(f"Unigram + bigram + balanced: "
      f"{f1_score(y_valid, best_valid_pred, average='macro'):.4f}")

Unigram + balanced: 0.4661
Unigram + bigram + balanced: 0.4712


In [12]:
error_analysis = valid_part[
    ["post_id", "text", "label"]
].copy()

error_analysis["prediction"] = best_model.predict(
    valid_part["clean"]
)

errors = error_analysis.loc[
    error_analysis["label"] != error_analysis["prediction"]
]

examples = errors.loc[
    (errors["label"] == "disgust")
    & (errors["prediction"] == "anger")
]

with pd.option_context("display.max_colwidth", None):
    display(examples.head(5))

,post_id,text,label,prediction
5,expfslt,"Takes a lot less effort to just say “fuck off, buy your own.”",disgust,anger
58,gfgm1tm,"Asmaerasmaer9441 I don't see any of you, the sons of the Gonder tribe, a platform for the unity of the elite and the conflict of identity. If you are finished, you can't leave your village of Gonder but you can only listen to me and you can create a new kind of hatred.",disgust,anger
247,3xz6olb,Please no body should tell me she was nervouslook at the confidence and agility she was speaking with dey shame her abeg,disgust,anger
611,mgvu4is,We cant even get sexually assaulted without them claiming we're gross fetishists,disgust,anger
656,v3z6bz8,"If I buy too much local icecream from the icecream store, Dryers will go bankrupt! (????) These people are so desperate.",disgust,anger


In the five validation samples where the true label was disgust but was predicted as anger, the text often contained abusive language, blame, sarcasm, or negative evaluations. These expressions may simultaneously convey anger and disgust, making the boundary between the two categories unclear. The TF-IDF word and word-phrase features fail to fully represent the context, irony, and speaker's stance, which may be one reason for the confusion; however, further examination of the model's feature weights is needed to confirm this.（文字說明曾使用 Google Translate 協助翻譯。）

### A2. Word + character TF-IDF 與參數比較 (Word + Character TF-IDF and Parameter Comparison)


In [13]:
from sklearn.pipeline import Pipeline, FeatureUnion

word_char_model = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
        )),
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=50000,
        )),
    ])),
    ("svm", LinearSVC(
        C=0.25,
        class_weight="balanced",
        random_state=SEED,
        max_iter=10000,
    )),
])

word_char_model.fit(
    train_part["clean"],
    train_part["label"],
)

word_char_pred = word_char_model.predict(
    valid_part["clean"]
)

word_char_f1 = f1_score(
    valid_part["label"],
    word_char_pred,
    average="macro",
)

print(f"Word TF-IDF baseline: "
      f"{f1_score(y_valid, best_valid_pred, average='macro'):.4f}")
print(f"Word + character TF-IDF: {word_char_f1:.4f}")

print(classification_report(
    valid_part["label"],
    word_char_pred,
    digits=4,
    zero_division=0,
))

Word TF-IDF baseline: 0.4712
Word + character TF-IDF: 0.5024
              precision    recall  f1-score   support

       anger     0.4422    0.4815    0.4610       135
     disgust     0.4225    0.3297    0.3704        91
        fear     0.5400    0.4655    0.5000        58
         joy     0.6301    0.6229    0.6264       175
     sadness     0.5407    0.5794    0.5594       126
    surprise     0.4742    0.5227    0.4973        88

    accuracy                         0.5201       673
   macro avg     0.5083    0.5003    0.5024       673
weighted avg     0.5195    0.5201    0.5183       673



In [14]:
from sklearn.model_selection import cross_val_score

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

models_to_compare = {
    "Word TF-IDF": best_model,
    "Word + character TF-IDF": word_char_model,
}

cv_rows = []

for name, model in models_to_compare.items():
    scores = cross_val_score(
        model,
        train["clean"],
        train["label"],
        groups=train["clean"],
        cv=cv,
        scoring="f1_macro",
        n_jobs=1,
    )

    print(name, "各折分數：", np.round(scores, 4))

    cv_rows.append({
        "model": name,
        "mean_macro_f1": scores.mean(),
        "std": scores.std(),
    })

display(pd.DataFrame(cv_rows).round(4))

Word TF-IDF 各折分數： [0.4712 0.5006 0.4923 0.4862 0.4834]
Word + character TF-IDF 各折分數： [0.5024 0.5111 0.5051 0.5115 0.5113]


,model,mean_macro_f1,std
0,Word TF-IDF,0.4867,0.0098
1,Word + character TF-IDF,0.5083,0.0038


In [15]:
from sklearn.base import clone
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedGroupKFold,
)

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

cv_splits = list(cv.split(
    train["clean"],
    train["label"],
    groups=train["clean"],
))

param_grid = {
    "features__word__min_df": [1, 2],
    "svm__C": [0.1, 0.25, 1.0],
}

search = GridSearchCV(
    estimator=clone(word_char_model),
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv_splits,
    n_jobs=1,
    refit=True,
    return_train_score=False,
    verbose=1,
)

search.fit(train["clean"], train["label"])

search_results = pd.DataFrame(search.cv_results_)

display(
    search_results[[
        "param_features__word__min_df",
        "param_svm__C",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]]
    .sort_values("rank_test_score")
    .round(4)
)

print("最佳參數：", search.best_params_)
print(f"最佳五折 Macro F1：{search.best_score_:.4f}")

tuned_word_char_model = search.best_estimator_

Fitting 5 folds for each of 6 candidates, totalling 30 fits


,param_features__word__min_df,param_svm__C,mean_test_score,std_test_score,rank_test_score
0,1,0.10,0.5244,0.0079,1
1,1,0.25,0.5233,0.0040,2
3,2,0.10,0.5152,0.0067,3
2,1,1.00,0.5139,0.0049,4
4,2,0.25,0.5083,0.0038,5
5,2,1.00,0.4902,0.0059,6


最佳參數： {'features__word__min_df': 1, 'svm__C': 0.1}
最佳五折 Macro F1：0.5244


### A3. 情緒模型直接推論（RoBERTa-large）(Direct Inference with Emotion Model (RoBERTa-large))


In [16]:
import sys
import os
import torch
import transformers
from transformers.utils import is_torch_available

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers sees torch:", is_torch_available())
print("USE_TORCH:", os.environ.get("USE_TORCH"))
print("USE_TF:", os.environ.get("USE_TF"))

Python: C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\.venv\Scripts\python.exe
PyTorch: 2.6.0+cu124
Transformers: 5.17.0
CUDA available: True
Transformers sees torch: True
USE_TORCH: None
USE_TF: None


In [17]:
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm

MODEL_NAME = "j-hartmann/emotion-english-roberta-large"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("執行裝置：", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
emotion_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
).to(device)
emotion_model.eval()


allowed_labels = sorted(train["label"].unique())

model_label_to_id = {
    label: int(idx)
    for idx, label in emotion_model.config.id2label.items()
}

assert set(allowed_labels).issubset(model_label_to_id)

allowed_ids = [
    model_label_to_id[label] for label in allowed_labels
]

@torch.inference_mode()
def predict_emotions(texts, batch_size=8):
    texts = list(texts)
    predictions = []

    for start in tqdm(range(0, len(texts), batch_size)):
        batch = texts[start:start + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(device)

        logits = emotion_model(**inputs).logits

        
        best_indices = logits[:, allowed_ids].argmax(dim=1).cpu().tolist()

        predictions.extend(
            allowed_labels[idx] for idx in best_indices
        )

    return np.array(predictions)



valid_predictions = predict_emotions(
    valid_part["text"].fillna("").astype(str)
)

score = f1_score(
    valid_part["label"],
    valid_predictions,
    average="macro",
)

print(f"\nPretrained model validation macro F1: {score:.4f}")
print(classification_report(
    valid_part["label"],
    valid_predictions,
    digits=4,
    zero_division=0,
))



test_predictions = predict_emotions(
    test["text"].fillna("").astype(str)
)


assert test["post_id"].is_unique

prediction_map = pd.Series(
    test_predictions,
    index=test["post_id"],
)


submission_pretrained = pd.DataFrame({
    "id": sample_submission["post_id"],
    "label": sample_submission["post_id"].map(prediction_map),
})

assert submission_pretrained.notna().all().all()
assert submission_pretrained["id"].is_unique
assert len(submission_pretrained) == len(test)
assert set(submission_pretrained["label"]).issubset(allowed_labels)

output_path = Path("submission_pretrained_roberta_large.csv")
submission_pretrained.to_csv(output_path, index=False)

print("\n提交檔已儲存：", output_path.resolve())
display(submission_pretrained.head())

執行裝置： cuda


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

  0%|          | 0/85 [00:00<?, ?it/s]


Pretrained model validation macro F1: 0.5715
              precision    recall  f1-score   support

       anger     0.5362    0.5481    0.5421       135
     disgust     0.4242    0.4615    0.4421        91
        fear     0.4865    0.6207    0.5455        58
         joy     0.8370    0.6457    0.7290       175
     sadness     0.7312    0.5397    0.6210       126
    surprise     0.4552    0.6932    0.5495        88

    accuracy                         0.5854       673
   macro avg     0.5784    0.5848    0.5715       673
weighted avg     0.6209    0.5854    0.5932       673



  0%|          | 0/180 [00:00<?, ?it/s]


提交檔已儲存： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\submission_pretrained_roberta_large.csv


,id,label
0,xyzx9jp,surprise
1,l3vzzxt,anger
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise


In [18]:
comparison = pd.DataFrame([
    {
        "model": "Word TF-IDF + SVM",
        "validation_macro_f1": f1_score(
            valid_part["label"],
            best_valid_pred,
            average="macro",
        ),
    },
    {
        "model": "Word + character TF-IDF + SVM",
        "validation_macro_f1": word_char_f1,
    },
    {
        "model": "Pretrained RoBERTa-large",
        "validation_macro_f1": score,
    },
])

display(comparison.round(4))

,model,validation_macro_f1
0,Word TF-IDF + SVM,0.4712
1,Word + character TF-IDF + SVM,0.5024
2,Pretrained RoBERTa-large,0.5715


### Fold 1：原版 lr=1e-5 重建步驟 (Fold 1: Reconstruction Steps (Original Version, lr=1e-5))



In [19]:
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 若先前已載入 large，先將它移回 CPU，釋放 GPU 空間。
if "emotion_model" in globals():
    emotion_model.cpu()
torch.cuda.empty_cache()

FT_MODEL_NAME = "j-hartmann/emotion-english-distilroberta-base"
device = torch.device("cuda")
MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 3

ft_tokenizer = AutoTokenizer.from_pretrained(FT_MODEL_NAME)
ft_model = AutoModelForSequenceClassification.from_pretrained(
    FT_MODEL_NAME
).to(device)

# 保留模型原有的標籤編碼，不重新初始化分類層。
ft_label_to_id = {
    label: int(idx)
    for idx, label in ft_model.config.id2label.items()
}

competition_labels = sorted(train["label"].unique())
assert set(competition_labels).issubset(ft_label_to_id)

competition_ids = [
    ft_label_to_id[label] for label in competition_labels
]


class EmotionDataset(Dataset):
    def __init__(self, frame):
        self.encodings = ft_tokenizer(
            frame["text"].fillna("").astype(str).tolist(),
            truncation=True,
            max_length=MAX_LENGTH,
        )
        self.targets = [
            ft_label_to_id[label] for label in frame["label"]
        ]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        item = {
            key: values[idx]
            for key, values in self.encodings.items()
        }
        item["labels"] = self.targets[idx]
        return item


collator = DataCollatorWithPadding(tokenizer=ft_tokenizer)

train_loader = DataLoader(
    EmotionDataset(train_part),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=0,
)

valid_loader = DataLoader(
    EmotionDataset(valid_part),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0,
)

print("模型：", FT_MODEL_NAME)
print("訓練裝置：", device)
print("資料準備完成")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

模型： j-hartmann/emotion-english-distilroberta-base
訓練裝置： cuda
資料準備完成


In [20]:
@torch.inference_mode()
def evaluate_ft_model():
    ft_model.eval()
    predictions = []

    for batch in valid_loader:
        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        logits = ft_model(**inputs).logits

        # 只從競賽六類選擇答案。
        indices = logits[:, competition_ids].argmax(dim=1)
        predictions.extend(
            competition_labels[idx]
            for idx in indices.cpu().tolist()
        )

    macro_f1 = f1_score(
        valid_part["label"],
        predictions,
        labels=competition_labels,
        average="macro",
        zero_division=0,
    )
    return macro_f1, predictions


# 包含 epoch 0，避免微調變差後仍覆蓋原本較好的模型。
best_f1, best_predictions = evaluate_ft_model()
best_epoch = 0
best_state = {
    key: value.detach().cpu().clone()
    for key, value in ft_model.state_dict().items()
}

history = [{
    "epoch": 0,
    "train_loss": np.nan,
    "validation_macro_f1": best_f1,
}]

print(f"未微調，128 tokens：{best_f1:.4f}")

optimizer = torch.optim.AdamW(
    ft_model.parameters(),
    lr=1e-5,
    weight_decay=0.01,
)
for epoch in range(1, EPOCHS + 1):
    ft_model.train()
    total_loss = 0.0
    total_samples = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for batch in progress:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        loss = ft_model(**batch).loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            ft_model.parameters(), max_norm=1.0
        )
        optimizer.step()

        batch_count = batch["labels"].size(0)
        total_loss += loss.item() * batch_count
        total_samples += batch_count

        progress.set_postfix(loss=f"{loss.item():.4f}")

    val_f1, val_predictions = evaluate_ft_model()
    average_loss = total_loss / total_samples

    history.append({
        "epoch": epoch,
        "train_loss": average_loss,
        "validation_macro_f1": val_f1,
    })

    print(
        f"Epoch {epoch}: "
        f"loss={average_loss:.4f}, "
        f"validation macro F1={val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        best_predictions = val_predictions
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in ft_model.state_dict().items()
        }

# 恢復最佳權重，存到本機專案資料夾。
ft_model.load_state_dict(best_state)
ft_model.eval()

save_dir = "models/distilroberta_best_validation"
ft_model.save_pretrained(save_dir)
ft_tokenizer.save_pretrained(save_dir)

ft_history = pd.DataFrame(history)
display(ft_history.round(4))

print(f"最佳 epoch：{best_epoch}")
print(f"最佳 validation macro F1：{best_f1:.4f}")
print("模型儲存位置：", save_dir)

print(classification_report(
    valid_part["label"],
    best_predictions,
    labels=competition_labels,
    digits=4,
    zero_division=0,
))

未微調，128 tokens：0.5454


Epoch 1/3:   0%|          | 0/336 [00:00<?, ?it/s]

Epoch 1: loss=1.1642, validation macro F1=0.5753


Epoch 2/3:   0%|          | 0/336 [00:00<?, ?it/s]

Epoch 2: loss=0.8998, validation macro F1=0.5886


Epoch 3/3:   0%|          | 0/336 [00:00<?, ?it/s]

Epoch 3: loss=0.7295, validation macro F1=0.5940


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,epoch,train_loss,validation_macro_f1
0,0,NaN,0.5454
1,1,1.1642,0.5753
2,2,0.8998,0.5886
3,3,0.7295,0.5940


最佳 epoch：3
最佳 validation macro F1：0.5940
模型儲存位置： models/distilroberta_best_validation
              precision    recall  f1-score   support

       anger     0.5664    0.6000    0.5827       135
     disgust     0.5690    0.3626    0.4430        91
        fear     0.5893    0.5690    0.5789        58
         joy     0.7283    0.7657    0.7465       175
     sadness     0.5652    0.7222    0.6341       126
    surprise     0.6479    0.5227    0.5786        88

    accuracy                         0.6211       673
   macro avg     0.6110    0.5904    0.5940       673
weighted avg     0.6212    0.6211    0.6152       673



### Fold 1：原版 lr=5e-6（最終融合使用） (Fold 1: Original version, lr=5e-6 (used in the final ensemble))


In [21]:
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 若先前已載入 large，先將它移回 CPU，釋放 GPU 空間。
if "emotion_model" in globals():
    emotion_model.cpu()
torch.cuda.empty_cache()

FT_MODEL_NAME = "j-hartmann/emotion-english-distilroberta-base"
device = torch.device("cuda")
MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 3

ft_tokenizer = AutoTokenizer.from_pretrained(FT_MODEL_NAME)
ft_model = AutoModelForSequenceClassification.from_pretrained(
    FT_MODEL_NAME
).to(device)

# 保留模型原有的標籤編碼，不重新初始化分類層。
ft_label_to_id = {
    label: int(idx)
    for idx, label in ft_model.config.id2label.items()
}

competition_labels = sorted(train["label"].unique())
assert set(competition_labels).issubset(ft_label_to_id)

competition_ids = [
    ft_label_to_id[label] for label in competition_labels
]


class EmotionDataset(Dataset):
    def __init__(self, frame):
        self.encodings = ft_tokenizer(
            frame["text"].fillna("").astype(str).tolist(),
            truncation=True,
            max_length=MAX_LENGTH,
        )
        self.targets = [
            ft_label_to_id[label] for label in frame["label"]
        ]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        item = {
            key: values[idx]
            for key, values in self.encodings.items()
        }
        item["labels"] = self.targets[idx]
        return item


collator = DataCollatorWithPadding(tokenizer=ft_tokenizer)

train_loader = DataLoader(
    EmotionDataset(train_part),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=0,
)

valid_loader = DataLoader(
    EmotionDataset(valid_part),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0,
)

print("模型：", FT_MODEL_NAME)
print("訓練裝置：", device)
print("資料準備完成")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

模型： j-hartmann/emotion-english-distilroberta-base
訓練裝置： cuda
資料準備完成


In [22]:
@torch.inference_mode()
def evaluate_ft_model():
    ft_model.eval()
    predictions = []

    for batch in valid_loader:
        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        logits = ft_model(**inputs).logits

        # 只從競賽六類選擇答案。
        indices = logits[:, competition_ids].argmax(dim=1)
        predictions.extend(
            competition_labels[idx]
            for idx in indices.cpu().tolist()
        )

    macro_f1 = f1_score(
        valid_part["label"],
        predictions,
        labels=competition_labels,
        average="macro",
        zero_division=0,
    )
    return macro_f1, predictions


# 包含 epoch 0，避免微調變差後仍覆蓋原本較好的模型。
best_f1, best_predictions = evaluate_ft_model()
best_epoch = 0
best_state = {
    key: value.detach().cpu().clone()
    for key, value in ft_model.state_dict().items()
}

history = [{
    "epoch": 0,
    "train_loss": np.nan,
    "validation_macro_f1": best_f1,
}]

print(f"未微調，128 tokens：{best_f1:.4f}")

optimizer = torch.optim.AdamW(
    ft_model.parameters(),
    lr=5e-6,
    weight_decay=0.01,
)
for epoch in range(1, EPOCHS + 1):
    ft_model.train()
    total_loss = 0.0
    total_samples = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for batch in progress:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        loss = ft_model(**batch).loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            ft_model.parameters(), max_norm=1.0
        )
        optimizer.step()

        batch_count = batch["labels"].size(0)
        total_loss += loss.item() * batch_count
        total_samples += batch_count

        progress.set_postfix(loss=f"{loss.item():.4f}")

    val_f1, val_predictions = evaluate_ft_model()
    average_loss = total_loss / total_samples

    history.append({
        "epoch": epoch,
        "train_loss": average_loss,
        "validation_macro_f1": val_f1,
    })

    print(
        f"Epoch {epoch}: "
        f"loss={average_loss:.4f}, "
        f"validation macro F1={val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        best_predictions = val_predictions
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in ft_model.state_dict().items()
        }

# 恢復最佳權重，存到本機專案資料夾。
ft_model.load_state_dict(best_state)
ft_model.eval()

save_dir = "models/distilroberta_lr5e6_unweighted"
ft_model.save_pretrained(save_dir)
ft_tokenizer.save_pretrained(save_dir)

ft_history = pd.DataFrame(history)
display(ft_history.round(4))

print(f"最佳 epoch：{best_epoch}")
print(f"最佳 validation macro F1：{best_f1:.4f}")
print("模型儲存位置：", save_dir)

print(classification_report(
    valid_part["label"],
    best_predictions,
    labels=competition_labels,
    digits=4,
    zero_division=0,
))

未微調，128 tokens：0.5454


Epoch 1/3:   0%|          | 0/336 [00:00<?, ?it/s]

Epoch 1: loss=1.2259, validation macro F1=0.5698


Epoch 2/3:   0%|          | 0/336 [00:00<?, ?it/s]

Epoch 2: loss=0.9839, validation macro F1=0.5788


Epoch 3/3:   0%|          | 0/336 [00:00<?, ?it/s]

Epoch 3: loss=0.8705, validation macro F1=0.5913


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,epoch,train_loss,validation_macro_f1
0,0,NaN,0.5454
1,1,1.2259,0.5698
2,2,0.9839,0.5788
3,3,0.8705,0.5913


最佳 epoch：3
最佳 validation macro F1：0.5913
模型儲存位置： models/distilroberta_lr5e6_unweighted
              precision    recall  f1-score   support

       anger     0.5442    0.5926    0.5674       135
     disgust     0.5636    0.3407    0.4247        91
        fear     0.6071    0.5862    0.5965        58
         joy     0.7090    0.7657    0.7363       175
     sadness     0.5867    0.6984    0.6377       126
    surprise     0.6316    0.5455    0.5854        88

    accuracy                         0.6166       673
   macro avg     0.6070    0.5882    0.5913       673
weighted avg     0.6145    0.6166    0.6100       673



In [23]:
# 單模型微調提交產生程式；歷史檔名不代表最佳 epoch。
# 修改訓練流程後若重跑，結果不等於歷史 Kaggle 成績。
from pathlib import Path

# 上一格已恢復最佳 epoch 的權重。
ft_model.eval()

@torch.inference_mode()
def predict_finetuned(texts):
    texts = list(texts)
    predictions = []

    for start in tqdm(range(0, len(texts), BATCH_SIZE)):
        inputs = ft_tokenizer(
            texts[start:start + BATCH_SIZE],
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)

        logits = ft_model(**inputs).logits
        indices = logits[:, competition_ids].argmax(dim=1)

        predictions.extend(
            competition_labels[idx]
            for idx in indices.cpu().tolist()
        )

    return predictions


ft_test_predictions = predict_finetuned(
    test["text"].fillna("").astype(str)
)

assert test["post_id"].is_unique

ft_prediction_map = pd.Series(
    ft_test_predictions,
    index=test["post_id"],
)

submission_ft = pd.DataFrame({
    "id": sample_submission["post_id"],
    "label": sample_submission["post_id"].map(ft_prediction_map),
})

assert submission_ft.notna().all().all()
assert submission_ft["id"].is_unique
assert len(submission_ft) == len(test)
assert set(submission_ft["label"]).issubset(competition_labels)

output_path = Path("submission_distilroberta_finetuned_epoch3.csv")
submission_ft.to_csv(output_path, index=False)

print("提交檔：", output_path.resolve())
display(submission_ft.head())

  0%|          | 0/180 [00:00<?, ?it/s]

提交檔： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\submission_distilroberta_finetuned_epoch3.csv


,id,label
0,xyzx9jp,surprise
1,l3vzzxt,anger
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise


In [24]:
lengths = [
    len(ids)
    for ids in ft_tokenizer(
        train["text"].fillna("").astype(str).tolist(),
        truncation=False,
        add_special_tokens=True,
    )["input_ids"]
]

lengths = pd.Series(lengths)

print(lengths.describe(percentiles=[0.90, 0.95, 0.99]))
print(f"超過 128 tokens：{(lengths > 128).mean():.2%}")
print(f"超過 256 tokens：{(lengths > 256).mean():.2%}")

count    3360.000000
mean       23.313095
std        30.154779
min         4.000000
90%        35.000000
95%        45.000000
99%       111.820000
max       511.000000
dtype: float64
超過 128 tokens：0.80%
超過 256 tokens：0.33%


### A4. 早期 Distil 學習率融合實驗 (Early Distil Learning Rate Fusion Experiments)


In [25]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm

# 確認兩個已訓練模型都存在。
model_paths = {
    "Original lr=1e-5": "models/distilroberta_best_validation",
    "Lower lr=5e-6": "models/distilroberta_lr5e6_unweighted",
}

for path in model_paths.values():
    assert Path(path, "config.json").exists(), f"找不到模型：{path}"

# 將先前模型移回 CPU，避免同時占用 GPU。
for name in ["ft_model", "full_model", "emotion_model"]:
    existing_model = globals().get(name)
    if existing_model is not None:
        existing_model.cpu()

gc.collect()
torch.cuda.empty_cache()

ensemble_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
ensemble_labels = sorted(train["label"].unique())


@torch.inference_mode()
def get_probabilities(model_path, texts):
    tokenizer = AutoTokenizer.from_pretrained(
        model_path, local_files_only=True
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, local_files_only=True
    ).to(ensemble_device)
    model.eval()

    # 各模型自行找標籤位置，統一輸出欄位順序。
    label_to_id = {
        label: int(idx)
        for idx, label in model.config.id2label.items()
    }
    allowed_ids = [label_to_id[label] for label in ensemble_labels]

    texts = list(texts)
    batches = []

    for start in tqdm(
        range(0, len(texts), 8),
        desc=Path(model_path).name,
    ):
        inputs = tokenizer(
            texts[start:start + 8],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(ensemble_device)

        logits = model(**inputs).logits

        # 排除 neutral，再對六類計算機率。
        probabilities = torch.softmax(
            logits[:, allowed_ids], dim=1
        )
        batches.append(probabilities.cpu().numpy())

    model.cpu()
    del model
    gc.collect()
    torch.cuda.empty_cache()

    return np.concatenate(batches, axis=0)


valid_texts = valid_part["text"].fillna("").astype(str)
valid_probabilities = {
    name: get_probabilities(path, valid_texts)
    for name, path in model_paths.items()
}

label_array = np.array(ensemble_labels)
comparison_rows = []

for name, probabilities in valid_probabilities.items():
    predictions = label_array[probabilities.argmax(axis=1)]
    comparison_rows.append({
        "model": name,
        "validation_macro_f1": f1_score(
            valid_part["label"],
            predictions,
            labels=ensemble_labels,
            average="macro",
            zero_division=0,
        ),
    })

# 固定各占一半。
ensemble_valid_probs = np.mean(
    list(valid_probabilities.values()), axis=0
)
ensemble_valid_pred = label_array[
    ensemble_valid_probs.argmax(axis=1)
]

ensemble_f1 = f1_score(
    valid_part["label"],
    ensemble_valid_pred,
    labels=ensemble_labels,
    average="macro",
    zero_division=0,
)

comparison_rows.append({
    "model": "50/50 probability ensemble",
    "validation_macro_f1": ensemble_f1,
})

display(pd.DataFrame(comparison_rows).round(4))

print(classification_report(
    valid_part["label"],
    ensemble_valid_pred,
    labels=ensemble_labels,
    digits=4,
    zero_division=0,
))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

distilroberta_best_validation:   0%|          | 0/85 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

distilroberta_lr5e6_unweighted:   0%|          | 0/85 [00:00<?, ?it/s]

,model,validation_macro_f1
0,Original lr=1e-5,0.5940
1,Lower lr=5e-6,0.5913
2,50/50 probability ensemble,0.5965


              precision    recall  f1-score   support

       anger     0.5634    0.5926    0.5776       135
     disgust     0.5714    0.3516    0.4354        91
        fear     0.5965    0.5862    0.5913        58
         joy     0.7204    0.7657    0.7424       175
     sadness     0.5796    0.7222    0.6431       126
    surprise     0.6400    0.5455    0.5890        88

    accuracy                         0.6226       673
   macro avg     0.6119    0.5940    0.5965       673
weighted avg     0.6212    0.6226    0.6162       673



## DeBERTa tokenizer

In [26]:
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "目前 kernel 沒有偵測到 CUDA。"
assert set(train_part["clean"]).isdisjoint(valid_part["clean"])

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DB_MODEL_NAME = "microsoft/deberta-v3-base"
DB_DEVICE = torch.device("cuda")

# 第一個實驗先用這組設定，不同時搜尋多組參數。
DB_MAX_LENGTH = 128
DB_BATCH_SIZE = 4
DB_ACCUM_STEPS = 4       # 有效 batch size = 4 × 4 = 16
DB_EPOCHS = 4
DB_LR = 2e-5
DB_SAVE_DIR = Path("models/deberta_v3_base_validation")

db_labels = sorted(train["label"].unique())
db_label2id = {
    label: idx for idx, label in enumerate(db_labels)
}
db_id2label = {
    idx: label for label, idx in db_label2id.items()
}

# 使用 DeBERTa 自己的 tokenizer。
db_tokenizer = AutoTokenizer.from_pretrained(
    DB_MODEL_NAME,
    use_fast=False,
)

# 新增六類分類層；不沿用 Hartmann 模型的七類編碼。
db_model = AutoModelForSequenceClassification.from_pretrained(
    DB_MODEL_NAME,
    num_labels=len(db_labels),
    label2id=db_label2id,
    id2label=db_id2label,
).float().to(DB_DEVICE)

print("模型參數型別：", next(db_model.parameters()).dtype)


class DebertaEmotionDataset(Dataset):
    def __init__(self, frame):
        self.encodings = db_tokenizer(
            frame["text"].fillna("").astype(str).tolist(),
            truncation=True,
            max_length=DB_MAX_LENGTH,
        )
        self.targets = [
            db_label2id[label] for label in frame["label"]
        ]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        item = {
            key: values[idx]
            for key, values in self.encodings.items()
        }
        item["labels"] = self.targets[idx]
        return item


db_collator = DataCollatorWithPadding(
    tokenizer=db_tokenizer,
    pad_to_multiple_of=8,
)

db_train_loader = DataLoader(
    DebertaEmotionDataset(train_part),
    batch_size=DB_BATCH_SIZE,
    shuffle=True,
    collate_fn=db_collator,
    num_workers=0,
)

db_valid_loader = DataLoader(
    DebertaEmotionDataset(valid_part),
    batch_size=DB_BATCH_SIZE,
    shuffle=False,
    collate_fn=db_collator,
    num_workers=0,
)

print("GPU:", torch.cuda.get_device_name(0))
print("訓練筆數:", len(train_part))
print("驗證筆數:", len(valid_part))
print("標籤對應:", db_label2id)
print("模型準備完成")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.den

模型參數型別： torch.float32
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU
訓練筆數: 2687
驗證筆數: 673
標籤對應: {'anger': 0, 'disgust': 1, 'fear': 2, 'joy': 3, 'sadness': 4, 'surprise': 5}
模型準備完成


In [27]:
import math

@torch.inference_mode()
def evaluate_deberta():
    db_model.eval()
    predictions = []

    for batch in db_valid_loader:
        inputs = {
            key: value.to(DB_DEVICE)
            for key, value in batch.items()
            if key != "labels"
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            logits = db_model(**inputs).logits

        indices = logits.argmax(dim=1).cpu().tolist()
        predictions.extend(db_id2label[idx] for idx in indices)

    score = f1_score(
        valid_part["label"],
        predictions,
        labels=db_labels,
        average="macro",
        zero_division=0,
    )
    return score, predictions


db_optimizer = torch.optim.AdamW(
    db_model.parameters(),
    lr=DB_LR,
    weight_decay=0.01,
)

updates_per_epoch = math.ceil(
    len(db_train_loader) / DB_ACCUM_STEPS
)
total_updates = updates_per_epoch * DB_EPOCHS

db_scheduler = get_linear_schedule_with_warmup(
    db_optimizer,
    num_warmup_steps=int(total_updates * 0.1),
    num_training_steps=total_updates,
)

db_scaler = torch.amp.GradScaler("cuda")

db_best_f1 = -1.0
db_best_epoch = None
db_history = []

for epoch in range(1, DB_EPOCHS + 1):
    db_model.train()
    db_optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0
    total_samples = 0

    progress = tqdm(
        db_train_loader,
        desc=f"DeBERTa epoch {epoch}/{DB_EPOCHS}",
    )

    for step, batch in enumerate(progress):
        batch = {
            key: value.to(DB_DEVICE)
            for key, value in batch.items()
        }

        # 最後不足四個 batch 的累積區段也正確縮放。
        group_start = (step // DB_ACCUM_STEPS) * DB_ACCUM_STEPS
        group_size = min(
            DB_ACCUM_STEPS,
            len(db_train_loader) - group_start,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            raw_loss = db_model(**batch).loss
            loss = raw_loss / group_size

        db_scaler.scale(loss).backward()

        batch_count = batch["labels"].size(0)
        total_loss += raw_loss.item() * batch_count
        total_samples += batch_count

        should_update = (
            (step + 1) % DB_ACCUM_STEPS == 0
            or step + 1 == len(db_train_loader)
        )

        if should_update:
            db_scaler.unscale_(db_optimizer)
            torch.nn.utils.clip_grad_norm_(
                db_model.parameters(),
                max_norm=1.0,
            )

            previous_scale = db_scaler.get_scale()
            db_scaler.step(db_optimizer)
            db_scaler.update()

            # 若溢位導致 optimizer 跳過更新，scheduler 也不前進。
            if db_scaler.get_scale() >= previous_scale:
                db_scheduler.step()

            db_optimizer.zero_grad(set_to_none=True)

        progress.set_postfix(loss=f"{raw_loss.item():.4f}")

    val_f1, val_predictions = evaluate_deberta()
    train_loss = total_loss / total_samples

    db_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_macro_f1": val_f1,
    })

    print(
        f"Epoch {epoch}: "
        f"loss={train_loss:.4f}, "
        f"validation macro F1={val_f1:.4f}"
    )

    if val_f1 > db_best_f1:
        db_best_f1 = val_f1
        db_best_epoch = epoch

        db_model.save_pretrained(DB_SAVE_DIR)
        db_tokenizer.save_pretrained(DB_SAVE_DIR)

        # 保存驗證預測，方便後續錯誤分析。
        validation_output = valid_part[
            ["post_id", "text", "label"]
        ].copy()
        validation_output["prediction"] = val_predictions
        validation_output.to_csv(
            DB_SAVE_DIR / "validation_predictions.csv",
            index=False,
        )

db_history = pd.DataFrame(db_history)
db_history.to_csv(DB_SAVE_DIR / "history.csv", index=False)

display(db_history.round(4))
print("最佳 epoch:", db_best_epoch)
print(f"最佳 validation macro F1: {db_best_f1:.4f}")
print("模型位置:", DB_SAVE_DIR.resolve())

DeBERTa epoch 1/4:   0%|          | 0/672 [00:00<?, ?it/s]

Epoch 1: loss=1.6518, validation macro F1=0.2479


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DeBERTa epoch 2/4:   0%|          | 0/672 [00:00<?, ?it/s]

Epoch 2: loss=1.2032, validation macro F1=0.5121


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

Epoch 4: loss=0.7039, validation macro F1=0.5680


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,epoch,train_loss,validation_macro_f1
0,1,1.6518,0.2479
1,2,1.2032,0.5121
2,3,0.8981,0.5366
3,4,0.7039,0.5680


最佳 epoch: 4
最佳 validation macro F1: 0.5680
模型位置: C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\deberta_v3_base_validation


In [28]:
import gc
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm

# 與剛才訓練時的儲存路徑一致。
DB_SAVE_DIR = Path("models/deberta_v3_base_validation")
assert (DB_SAVE_DIR / "config.json").exists(), "找不到已保存的模型。"

# 釋放訓練時留下的 GPU 記憶體。
for name in ["db_optimizer", "db_scheduler", "db_scaler"]:
    globals().pop(name, None)

if "db_model" in globals():
    db_model.cpu()
    del db_model

gc.collect()
torch.cuda.empty_cache()

DB_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
DB_BATCH_SIZE = 4
DB_MAX_LENGTH = 128

# 從本機載入最佳 checkpoint，而非原始pretrain model。
db_tokenizer = AutoTokenizer.from_pretrained(DB_SAVE_DIR)
db_model = AutoModelForSequenceClassification.from_pretrained(
    DB_SAVE_DIR
).to(DB_DEVICE)
db_model.eval()

# 使用保存於模型中的標籤對應。
db_id2label = {
    int(idx): label
    for idx, label in db_model.config.id2label.items()
}
db_labels = [
    db_id2label[idx]
    for idx in range(db_model.config.num_labels)
]

assert set(db_labels) == set(train["label"].unique())

print("執行裝置：", DB_DEVICE)
print("載入模型：", DB_SAVE_DIR.resolve())
print("標籤順序：", db_labels)


@torch.inference_mode()
def predict_deberta_saved(texts):
    texts = list(texts)
    predictions = []

    for start in tqdm(
        range(0, len(texts), DB_BATCH_SIZE),
        desc="DeBERTa prediction",
    ):
        inputs = db_tokenizer(
            texts[start:start + DB_BATCH_SIZE],
            padding=True,
            truncation=True,
            max_length=DB_MAX_LENGTH,
            return_tensors="pt",
        ).to(DB_DEVICE)

        # GPU 使用與訓練驗證時相同的混合精度。
        if DB_DEVICE.type == "cuda":
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                logits = db_model(**inputs).logits
        else:
            logits = db_model(**inputs).logits

        indices = logits.argmax(dim=1).cpu().tolist()
        predictions.extend(db_id2label[idx] for idx in indices)

    return predictions


# 1. 確認最佳模型的驗證表現。
db_valid_pred = predict_deberta_saved(
    valid_part["text"].fillna("").astype(str)
)

db_validation_f1 = f1_score(
    valid_part["label"],
    db_valid_pred,
    labels=db_labels,
    average="macro",
    zero_division=0,
)

print(f"\n最佳模型 validation macro F1：{db_validation_f1:.4f}")
print(classification_report(
    valid_part["label"],
    db_valid_pred,
    labels=db_labels,
    digits=4,
    zero_division=0,
))


# 2. 預測競賽測試資料。
db_test_pred = predict_deberta_saved(
    test["text"].fillna("").astype(str)
)

assert test["post_id"].is_unique

db_prediction_map = pd.Series(
    db_test_pred,
    index=test["post_id"],
)

# 依照 sample_submission 的 ID 順序排列，
# 延續先前已成功提交的 id、label 格式。
submission_deberta = pd.DataFrame({
    "id": sample_submission["post_id"].to_numpy(),
    "label": sample_submission["post_id"].map(
        db_prediction_map
    ).to_numpy(),
})

assert submission_deberta.notna().all().all()
assert submission_deberta["id"].is_unique
assert len(submission_deberta) == len(test)
assert set(submission_deberta["label"]).issubset(db_labels)

db_output_path = Path("submission_deberta_v3_base_best.csv")
submission_deberta.to_csv(db_output_path, index=False)

print("\n提交筆數：", len(submission_deberta))
print("CSV 位置：", db_output_path.resolve())
display(submission_deberta.head())

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

執行裝置： cuda
載入模型： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\deberta_v3_base_validation
標籤順序： ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']


DeBERTa prediction:   0%|          | 0/169 [00:00<?, ?it/s]


最佳模型 validation macro F1：0.5680
              precision    recall  f1-score   support

       anger     0.5766    0.5852    0.5809       135
     disgust     0.5088    0.3187    0.3919        91
        fear     0.6078    0.5345    0.5688        58
         joy     0.7204    0.7657    0.7424       175
     sadness     0.5455    0.5714    0.5581       126
    surprise     0.5091    0.6364    0.5657        88

    accuracy                         0.5958       673
   macro avg     0.5780    0.5686    0.5680       673
weighted avg     0.5929    0.5958    0.5900       673



DeBERTa prediction:   0%|          | 0/360 [00:00<?, ?it/s]


提交筆數： 1440
CSV 位置： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\submission_deberta_v3_base_best.csv


,id,label
0,xyzx9jp,surprise
1,l3vzzxt,anger
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise


### B1. Fold 1 驗證與單組分割融合 (B1. Fold 1 Validation and Single-Fold Split Fusion)


In [29]:
import gc
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

ROBERTA_DIR = "models/distilroberta_lr5e6_unweighted"
DEBERTA_DIR = "models/deberta_v3_base_validation"

# 沿用先前訓練時的截斷長度。
MAX_LENGTH = 128
BATCH_SIZE = 4
eval_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 統一兩個模型的機率欄位順序。
ensemble_labels = sorted(train["label"].unique())
valid_texts = valid_part["text"].fillna("").astype(str).tolist()
valid_targets = valid_part["label"].to_numpy()

assert len(valid_texts) > 0
assert set(train_part["clean"]).isdisjoint(valid_part["clean"])

for model_dir in [ROBERTA_DIR, DEBERTA_DIR]:
    assert (Path(model_dir) / "config.json").exists(), (
        f"找不到模型：{model_dir}"
    )

print("執行裝置：", eval_device)
print("驗證筆數：", len(valid_texts))
print("統一標籤順序：", ensemble_labels)


@torch.inference_mode()
def get_validation_probabilities(model_dir):
    # 一次只載入一個模型，減少 GPU 記憶體需求。
    tokenizer_local = AutoTokenizer.from_pretrained(model_dir)
    model_local = (
        AutoModelForSequenceClassification
        .from_pretrained(model_dir)
        .to(eval_device)
    )
    model_local.eval()

    label_to_id = {
        str(label).lower(): int(idx)
        for idx, label in model_local.config.id2label.items()
    }

    assert set(ensemble_labels).issubset(label_to_id), (
        f"{model_dir} 的標籤無法對齊：{label_to_id}"
    )

    selected_ids = [
        label_to_id[label] for label in ensemble_labels
    ]

    batches = []

    try:
        for start in tqdm(
            range(0, len(valid_texts), BATCH_SIZE),
            desc=Path(model_dir).name,
        ):
            inputs = tokenizer_local(
                valid_texts[start:start + BATCH_SIZE],
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            ).to(eval_device)

            logits = model_local(**inputs).logits

            # 先取出競賽六類，再轉成加總為 1 的機率。
            # 若 RoBERTa 還有 neutral，這裡會排除它。
            probabilities = torch.softmax(
                logits[:, selected_ids].float(),
                dim=1,
            )
            batches.append(probabilities.cpu().numpy())

        return np.concatenate(batches, axis=0)

    finally:
        model_local.cpu()
        del model_local, tokenizer_local
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


roberta_valid_probs = get_validation_probabilities(ROBERTA_DIR)
deberta_valid_probs = get_validation_probabilities(DEBERTA_DIR)

label_array = np.array(ensemble_labels)

def evaluate_probabilities(probabilities):
    predictions = label_array[probabilities.argmax(axis=1)]
    score = f1_score(
        valid_targets,
        predictions,
        labels=ensemble_labels,
        average="macro",
        zero_division=0,
    )
    return score


# 少量固定比例比較，包含兩個單模型作為參考。
results = []

for roberta_weight in [1.0, 0.75, 0.50, 0.0]:
    blended_probs = (
        roberta_weight * roberta_valid_probs
        + (1.0 - roberta_weight) * deberta_valid_probs
    )

    results.append({
        "RoBERTa 比例": roberta_weight,
        "DeBERTa 比例": 1.0 - roberta_weight,
        "validation_macro_f1": evaluate_probabilities(blended_probs),
    })

ensemble_comparison = pd.DataFrame(results)

display(ensemble_comparison.round(4))

roberta_baseline = ensemble_comparison.loc[
    ensemble_comparison["RoBERTa 比例"] == 1.0,
    "validation_macro_f1",
].iloc[0]

ensemble_comparison["相對 RoBERTa 的改善"] = (
    ensemble_comparison["validation_macro_f1"]
    - roberta_baseline
)

print("\n依驗證分數排序：")
display(
    ensemble_comparison.sort_values(
        "validation_macro_f1",
        ascending=False,
    ).round(4)
)

執行裝置： cuda
驗證筆數： 673
統一標籤順序： ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

distilroberta_lr5e6_unweighted:   0%|          | 0/169 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

deberta_v3_base_validation:   0%|          | 0/169 [00:00<?, ?it/s]

,RoBERTa 比例,DeBERTa 比例,validation_macro_f1
0,1.00,0.00,0.5913
1,0.75,0.25,0.6059
2,0.50,0.50,0.5965
3,0.00,1.00,0.5680



依驗證分數排序：


,RoBERTa 比例,DeBERTa 比例,validation_macro_f1,相對 RoBERTa 的改善
1,0.75,0.25,0.6059,0.0146
2,0.50,0.50,0.5965,0.0052
0,1.00,0.00,0.5913,0.0000
3,0.00,1.00,0.5680,-0.0233


In [30]:
# 沿用上一格的函式，但將輸入切換為測試集。
# finally 會恢復 valid_texts，避免影響後續驗證。
original_valid_texts = valid_texts

try:
    valid_texts = (
        test["text"].fillna("").astype(str).tolist()
    )

    roberta_test_probs = get_validation_probabilities(
        ROBERTA_DIR
    )
    deberta_test_probs = get_validation_probabilities(
        DEBERTA_DIR
    )
finally:
    valid_texts = original_valid_texts

# 與驗證時相同：六類機率各占 50%。
ensemble_test_probs = (
    0.5 * roberta_test_probs
    + 0.5 * deberta_test_probs
)

ensemble_test_predictions = np.array(ensemble_labels)[
    ensemble_test_probs.argmax(axis=1)
]

# 依 post_id 對齊官方提交範例的順序。
assert test["post_id"].is_unique

prediction_map = pd.Series(
    ensemble_test_predictions,
    index=test["post_id"].astype(str),
)

submission_ids = sample_submission["post_id"].astype(str)

submission_ensemble = pd.DataFrame({
    "id": submission_ids.to_numpy(),
    "label": submission_ids.map(prediction_map).to_numpy(),
})

assert len(submission_ensemble) == len(test)
assert submission_ensemble["id"].is_unique
assert submission_ensemble.notna().all().all()
assert set(submission_ensemble["label"]).issubset(
    ensemble_labels
)

output_path = Path("submission_roberta_deberta_50_50.csv")
submission_ensemble.to_csv(output_path, index=False)

print("提交筆數：", len(submission_ensemble))
print("檔案位置：", output_path.resolve())
display(submission_ensemble.head())

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

distilroberta_lr5e6_unweighted:   0%|          | 0/360 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

deberta_v3_base_validation:   0%|          | 0/360 [00:00<?, ?it/s]

提交筆數： 1440
檔案位置： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\submission_roberta_deberta_50_50.csv


,id,label
0,xyzx9jp,surprise
1,l3vzzxt,anger
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise


In [31]:
# 1. 確保函式這次讀取的是驗證集，而非測試集。
valid_texts = (
    valid_part["text"].fillna("").astype(str).tolist()
)

# 兩個模型都必須沒有使用 valid_part 訓練過。
distil_valid_probs = get_validation_probabilities(
    ROBERTA_DIR
)
deberta_valid_probs = get_validation_probabilities(
    DEBERTA_DIR
)

label_array = np.array(ensemble_labels)
expected_shape = (len(valid_part), len(ensemble_labels))

assert distil_valid_probs.shape == expected_shape
assert deberta_valid_probs.shape == expected_shape

for probabilities in [distil_valid_probs, deberta_valid_probs]:
    assert np.isfinite(probabilities).all()
    assert (probabilities >= 0).all()
    assert np.allclose(
        probabilities.sum(axis=1), 1.0, atol=1e-3
    )

# 2. 比較少量固定比例，包含兩個單模型作參考。
ratio_rows = []

for distil_weight in [0.0, 0.25, 0.50, 0.75, 1.0]:
    probabilities = (
        distil_weight * distil_valid_probs
        + (1.0 - distil_weight) * deberta_valid_probs
    )

    predictions = label_array[
        probabilities.argmax(axis=1)
    ]

    ratio_rows.append({
        "DistilRoBERTa_weight": distil_weight,
        "DeBERTa_weight": 1.0 - distil_weight,
        "validation_macro_f1": f1_score(
            valid_part["label"],
            predictions,
            labels=ensemble_labels,
            average="macro",
            zero_division=0,
        ),
    })

ratio_results = pd.DataFrame(ratio_rows)

display(
    ratio_results.sort_values(
        "validation_macro_f1", ascending=False
    ).round(4)
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

distilroberta_lr5e6_unweighted:   0%|          | 0/169 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

deberta_v3_base_validation:   0%|          | 0/169 [00:00<?, ?it/s]

,DistilRoBERTa_weight,DeBERTa_weight,validation_macro_f1
3,0.75,0.25,0.6059
2,0.50,0.50,0.5965
4,1.00,0.00,0.5913
1,0.25,0.75,0.5690
0,0.00,1.00,0.5680


## 最終提交二：全資料 seed 42，50/50 (Final Submission 2: Full dataset, seed 42, 50/50 split.)

正式選取版本，Public 0.6246、Private 0.6160。

Selected as the official submission: Public 0.6246, Private 0.6160.

In [32]:
## Merge
import gc
import math
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "請先確認目前 Kernel 可以使用 CUDA。"
assert len(train) == 3360
assert len(test) == 1440
assert train["post_id"].is_unique
assert test["post_id"].is_unique
assert train["label"].notna().all()

FULL_DEVICE = torch.device("cuda")
FULL_LABELS = sorted(train["label"].unique())
assert len(FULL_LABELS) == 6

# 每次實驗使用不同資料夾，保留既有模型與 CSV。
FULL_RUN_DIR = (
    Path("models")
    / f"fulltrain_ensemble_{datetime.now():%Y%m%d_%H%M%S}"
)
FULL_RUN_DIR.mkdir(parents=True, exist_ok=False)

print("GPU：", torch.cuda.get_device_name(0))
print("本次儲存位置：", FULL_RUN_DIR.resolve())


def reset_full_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class FullEmotionDataset(Dataset):
    def __init__(self, frame, tokenizer, label_to_id):
        self.encodings = tokenizer(
            frame["text"].fillna("").astype(str).tolist(),
            truncation=True,
            max_length=128,
        )
        self.targets = [
            label_to_id[label] for label in frame["label"]
        ]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        item = {
            key: values[index]
            for key, values in self.encodings.items()
        }
        item["labels"] = self.targets[index]
        return item


@torch.inference_mode()
def full_test_probabilities(model, tokenizer):
    model.eval()

    label_to_id = {
        str(label).lower(): int(index)
        for index, label in model.config.id2label.items()
    }
    selected_ids = [
        label_to_id[label] for label in FULL_LABELS
    ]

    texts = test["text"].fillna("").astype(str).tolist()
    batches = []

    # 沿用原本融合程式的 FP32 推論方式。
    for start in tqdm(
        range(0, len(texts), 4),
        desc="Test prediction",
    ):
        inputs = tokenizer(
            texts[start:start + 4],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(FULL_DEVICE)

        logits = model(**inputs).logits

        # 先排除 neutral，再對競賽六類計算機率。
        probabilities = torch.softmax(
            logits[:, selected_ids].float(),
            dim=1,
        )
        batches.append(probabilities.cpu().numpy())

    return np.concatenate(batches, axis=0)


def train_full_model(kind, seed=42):
    reset_full_seed(seed)

    is_deberta = kind == "deberta"
    assert kind in {"distilroberta", "deberta"}

    if is_deberta:
        model_name = "microsoft/deberta-v3-base"
        epochs = 3
        batch_size = 4
        accum_steps = 4
        learning_rate = 2e-5

        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            use_fast=False,
        )
        label_to_id = {
            label: index
            for index, label in enumerate(FULL_LABELS)
        }
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=len(FULL_LABELS),
            label2id=label_to_id,
            id2label={
                index: label
                for label, index in label_to_id.items()
            },
        ).float().to(FULL_DEVICE)

    else:
        model_name = (
            "j-hartmann/emotion-english-distilroberta-base"
        )
        epochs = 2
        batch_size = 8
        accum_steps = 1
        learning_rate = 5e-6

        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name
        ).float().to(FULL_DEVICE)

        # 保留原本七類分類層與其標籤編碼。
        label_to_id = {
            label: int(index)
            for index, label in model.config.id2label.items()
        }

    assert set(FULL_LABELS).issubset(label_to_id)

    collator_options = (
        {"pad_to_multiple_of": 8} if is_deberta else {}
    )
    loader = DataLoader(
        FullEmotionDataset(train, tokenizer, label_to_id),
        batch_size=batch_size,
        shuffle=True,
        collate_fn=DataCollatorWithPadding(
            tokenizer=tokenizer,
            **collator_options,
        ),
        num_workers=0,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=0.01,
    )

    scheduler = None
    scaler = None

    if is_deberta:
        # 原實驗排定 4 輪，最佳 checkpoint 在第 3 輪。
        total_updates = math.ceil(
            len(loader) / accum_steps
        ) * 4

        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(total_updates * 0.1),
            num_training_steps=total_updates,
        )
        scaler = torch.amp.GradScaler("cuda")

    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        total_loss = 0.0
        total_samples = 0

        progress = tqdm(
            loader,
            desc=f"{kind}: epoch {epoch}/{epochs}",
        )

        for step, batch in enumerate(progress):
            batch = {
                key: value.to(FULL_DEVICE)
                for key, value in batch.items()
            }

            if is_deberta:
                group_start = (
                    step // accum_steps
                ) * accum_steps
                group_size = min(
                    accum_steps,
                    len(loader) - group_start,
                )

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    raw_loss = model(**batch).loss
                    loss = raw_loss / group_size

                if not torch.isfinite(raw_loss).item():
                    raise RuntimeError("訓練 loss 非有限值，已停止。")

                scaler.scale(loss).backward()

                should_update = (
                    (step + 1) % accum_steps == 0
                    or step + 1 == len(loader)
                )

                if should_update:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(), 1.0
                    )

                    previous_scale = scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()

                    if scaler.get_scale() >= previous_scale:
                        scheduler.step()

                    optimizer.zero_grad(set_to_none=True)

            else:
                raw_loss = model(**batch).loss

                if not torch.isfinite(raw_loss).item():
                    raise RuntimeError("訓練 loss 非有限值，已停止。")

                raw_loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), 1.0
                )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            count = batch["labels"].size(0)
            total_loss += raw_loss.item() * count
            total_samples += count
            progress.set_postfix(
                loss=f"{raw_loss.item():.4f}"
            )

        average_loss = total_loss / total_samples
        history.append({
            "epoch": epoch,
            "train_loss": average_loss,
        })
        print(
            f"{kind} epoch {epoch}: "
            f"train_loss={average_loss:.4f}"
        )

    save_dir = FULL_RUN_DIR / kind
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    pd.DataFrame(history).to_csv(
        save_dir / "training_history.csv",
        index=False,
    )

    # 訓練完成，先釋放 optimizer，再進行推論。
    del optimizer, scheduler, scaler, loader
    gc.collect()
    torch.cuda.empty_cache()

    probabilities = full_test_probabilities(model, tokenizer)

    np.savez_compressed(
        save_dir / "test_probabilities.npz",
        probabilities=probabilities,
        post_id=test["post_id"].astype(str).to_numpy(),
        labels=np.array(FULL_LABELS),
    )

    model.cpu()
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    print("已保存：", save_dir.resolve())
    return probabilities


print("共用程式準備完成")

GPU： NVIDIA GeForce RTX 3070 Ti Laptop GPU
本次儲存位置： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\fulltrain_ensemble_20261125_215020
共用程式準備完成


In [33]:
full_distil_probs = train_full_model("distilroberta")

print("DistilRoBERTa 完成：", full_distil_probs.shape)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

distilroberta: epoch 1/2:   0%|          | 0/420 [00:00<?, ?it/s]

distilroberta epoch 1: train_loss=1.2048


distilroberta: epoch 2/2:   0%|          | 0/420 [00:00<?, ?it/s]

distilroberta epoch 2: train_loss=0.9848


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Test prediction:   0%|          | 0/360 [00:00<?, ?it/s]

已保存： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\fulltrain_ensemble_20261125_215020\distilroberta
DistilRoBERTa 完成： (1440, 6)


In [34]:
full_deberta_probs = train_full_model("deberta")

print("DeBERTa 完成：", full_deberta_probs.shape)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.den

deberta: epoch 1/3:   0%|          | 0/840 [00:00<?, ?it/s]

deberta epoch 1: train_loss=1.6409


deberta: epoch 2/3:   0%|          | 0/840 [00:00<?, ?it/s]

deberta epoch 2: train_loss=1.1581


deberta: epoch 3/3:   0%|          | 0/840 [00:00<?, ?it/s]

deberta epoch 3: train_loss=0.8646


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Test prediction:   0%|          | 0/360 [00:00<?, ?it/s]

已保存： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\fulltrain_ensemble_20261125_215020\deberta
DeBERTa 完成： (1440, 6)


In [35]:
## Submission
expected_shape = (len(test), len(FULL_LABELS))

for probabilities in [
    full_distil_probs,
    full_deberta_probs,
]:
    assert probabilities.shape == expected_shape
    assert np.isfinite(probabilities).all()
    assert (probabilities >= 0).all()
    assert np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=1e-4,
    )

full_ensemble_probs = (
    0.5 * full_distil_probs
    + 0.5 * full_deberta_probs
)

full_predictions = np.array(FULL_LABELS)[
    full_ensemble_probs.argmax(axis=1)
]

test_ids = test["post_id"].astype(str)
submission_ids = sample_submission["post_id"].astype(str)

assert test_ids.is_unique
assert submission_ids.is_unique
assert set(test_ids) == set(submission_ids)

prediction_map = pd.Series(
    full_predictions,
    index=test_ids,
)

submission_full = pd.DataFrame({
    "id": submission_ids.to_numpy(),
    "label": submission_ids.map(prediction_map).to_numpy(),
})

assert len(submission_full) == 1440
assert submission_full.notna().all().all()
assert set(submission_full["label"]).issubset(FULL_LABELS)

full_output_path = FULL_RUN_DIR / (
    "submission_fulltrain_distil_deberta_50_50.csv"
)
submission_full.to_csv(full_output_path, index=False)

print("提交筆數：", len(submission_full))
print("CSV 位置：", full_output_path.resolve())
display(submission_full.head())
print("\n預測類別分布：")
print(submission_full["label"].value_counts())

提交筆數： 1440
CSV 位置： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\fulltrain_ensemble_20261125_215020\submission_fulltrain_distil_deberta_50_50.csv


,id,label
0,xyzx9jp,surprise
1,l3vzzxt,anger
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise



預測類別分布：
label
joy         390
anger       360
sadness     261
surprise    207
disgust     118
fear        104
Name: count, dtype: int64


## 補充實驗：全資料多 seed 融合 (Supplementary Experiment: Fusion of Full Data with Multiple Seeds)


In [36]:
## Second seed
# 保留目前 Public 0.6246 那一版的預測機率。
original_full_distil_probs = full_distil_probs.copy()
original_full_deberta_probs = full_deberta_probs.copy()
original_full_labels = list(FULL_LABELS)
original_full_test_ids = test["post_id"].astype(str).to_numpy().copy()

backup_path = Path("fulltrain_original_predictions_backup.npz")

# 避免覆蓋已有備份。
if not backup_path.exists():
    np.savez_compressed(
        backup_path,
        distil_probs=original_full_distil_probs,
        deberta_probs=original_full_deberta_probs,
        labels=np.array(original_full_labels),
        post_id=original_full_test_ids,
    )

# 新模型使用獨立資料夾，不覆蓋舊模型。
FULL_RUN_DIR = (
    Path("models")
    / f"fulltrain_seed123_{datetime.now():%Y%m%d_%H%M%S_%f}"
)
FULL_RUN_DIR.mkdir(parents=True, exist_ok=False)

print("新模型位置：", FULL_RUN_DIR.resolve())
print("舊預測備份：", backup_path.resolve())

新模型位置： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\fulltrain_seed123_20261125_215640_733893
舊預測備份： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\fulltrain_original_predictions_backup.npz


In [37]:
seed123_distil_probs = train_full_model(
    "distilroberta",
    seed=123,
)
print("Seed 123 DistilRoBERTa 完成：", seed123_distil_probs.shape)

seed123_deberta_probs = train_full_model(
    "deberta",
    seed=123,
)
print("Seed 123 DeBERTa 完成：", seed123_deberta_probs.shape)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

distilroberta: epoch 1/2:   0%|          | 0/420 [00:00<?, ?it/s]

distilroberta epoch 1: train_loss=1.2144


distilroberta: epoch 2/2:   0%|          | 0/420 [00:00<?, ?it/s]

distilroberta epoch 2: train_loss=0.9905


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (9 outputs left out)

deberta epoch 3: train_loss=0.8710


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Test prediction:   0%|          | 0/360 [00:00<?, ?it/s]

已保存： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\fulltrain_seed123_20261125_215640_733893\deberta
Seed 123 DeBERTa 完成： (1440, 6)


In [38]:
# 確認標籤與測試資料順序沒有改變。
assert list(FULL_LABELS) == original_full_labels
assert np.array_equal(
    test["post_id"].astype(str).to_numpy(),
    original_full_test_ids,
)

probability_arrays = [
    original_full_distil_probs,
    original_full_deberta_probs,
    seed123_distil_probs,
    seed123_deberta_probs,
]

expected_shape = (len(test), len(FULL_LABELS))

for probabilities in probability_arrays:
    assert probabilities.shape == expected_shape
    assert np.isfinite(probabilities).all()
    assert (probabilities >= 0).all()
    assert np.allclose(
        probabilities.sum(axis=1), 1.0, atol=1e-3
    )

# 各架構內平均兩個 seed，再將兩個架構 50/50 融合。
multiseed_distil_probs = (
    original_full_distil_probs + seed123_distil_probs
) / 2

multiseed_deberta_probs = (
    original_full_deberta_probs + seed123_deberta_probs
) / 2

multiseed_probs = (
    0.5 * multiseed_distil_probs
    + 0.5 * multiseed_deberta_probs
)

label_array = np.array(FULL_LABELS)
multiseed_predictions = label_array[
    multiseed_probs.argmax(axis=1)
]

# 與目前 0.6246 版本比較預測改變量。
original_probs = (
    0.5 * original_full_distil_probs
    + 0.5 * original_full_deberta_probs
)
original_predictions = label_array[
    original_probs.argmax(axis=1)
]

changed = multiseed_predictions != original_predictions
print(f"與原版不同的預測：{changed.sum()} / {len(test)}")
print(f"改變比例：{changed.mean():.2%}")

prediction_map = pd.Series(
    multiseed_predictions,
    index=original_full_test_ids,
)

submission_ids = sample_submission["post_id"].astype(str)

submission_multiseed = pd.DataFrame({
    "id": submission_ids.to_numpy(),
    "label": submission_ids.map(prediction_map).to_numpy(),
})

assert submission_multiseed.notna().all().all()
assert submission_multiseed["id"].is_unique
assert len(submission_multiseed) == len(test)
assert set(submission_multiseed["label"]).issubset(FULL_LABELS)

output_path = Path(
    "submission_fulltrain_multiseed_distil_deberta_50_50.csv"
)
submission_multiseed.to_csv(output_path, index=False)

np.savez_compressed(
    FULL_RUN_DIR / "multiseed_test_probabilities.npz",
    probabilities=multiseed_probs,
    labels=label_array,
    post_id=original_full_test_ids,
)

print("新提交檔：", output_path.resolve())
display(submission_multiseed.head())

與原版不同的預測：92 / 1440
改變比例：6.39%
新提交檔： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\submission_fulltrain_multiseed_distil_deberta_50_50.csv


,id,label
0,xyzx9jp,surprise
1,l3vzzxt,anger
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise


## 補充實驗：類別加權 loss (Supplementary Experiment: Class-Weighted Loss)


In [39]:
## DistilRoBERTa 改用加權 loss

# 新實驗：釋放舊模型的 GPU 記憶體。
import gc
import torch

for old_name in ["emotion_model", "ft_model", "db_model"]:
    old_model = globals().get(old_name)
    if isinstance(old_model, torch.nn.Module):
        old_model.cpu()
del old_model

# 舊訓練 optimizer 可能仍持有 GPU tensor。
for old_name in ["optimizer", "db_optimizer"]:
    globals().pop(old_name, None)

gc.collect()
torch.cuda.empty_cache()

# 本次只改 class weights，固定相同 seed。
SEED = 42
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 若先前已載入 large，先將它移回 CPU，釋放 GPU 空間。
if "emotion_model" in globals():
    emotion_model.cpu()
torch.cuda.empty_cache()

FT_MODEL_NAME = "j-hartmann/emotion-english-distilroberta-base"
device = torch.device("cuda")
MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 3

ft_tokenizer = AutoTokenizer.from_pretrained(FT_MODEL_NAME)
ft_model = AutoModelForSequenceClassification.from_pretrained(
    FT_MODEL_NAME
).to(device)

# 保留模型原有的標籤編碼，不重新初始化分類層。
ft_label_to_id = {
    label: int(idx)
    for idx, label in ft_model.config.id2label.items()
}

competition_labels = sorted(train["label"].unique())
assert set(competition_labels).issubset(ft_label_to_id)

competition_ids = [
    ft_label_to_id[label] for label in competition_labels
]


class EmotionDataset(Dataset):
    def __init__(self, frame):
        self.encodings = ft_tokenizer(
            frame["text"].fillna("").astype(str).tolist(),
            truncation=True,
            max_length=MAX_LENGTH,
        )
        self.targets = [
            ft_label_to_id[label] for label in frame["label"]
        ]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        item = {
            key: values[idx]
            for key, values in self.encodings.items()
        }
        item["labels"] = self.targets[idx]
        return item


collator = DataCollatorWithPadding(tokenizer=ft_tokenizer)

train_loader = DataLoader(
    EmotionDataset(train_part),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=0,
)

valid_loader = DataLoader(
    EmotionDataset(valid_part),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0,
)

print("模型：", FT_MODEL_NAME)
print("訓練裝置：", device)
print("資料準備完成")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

模型： j-hartmann/emotion-english-distilroberta-base
訓練裝置： cuda
資料準備完成


In [40]:
@torch.inference_mode()
def evaluate_ft_model():
    ft_model.eval()
    predictions = []

    for batch in valid_loader:
        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        logits = ft_model(**inputs).logits

        # 只從競賽六類選擇答案。
        indices = logits[:, competition_ids].argmax(dim=1)
        predictions.extend(
            competition_labels[idx]
            for idx in indices.cpu().tolist()
        )

    macro_f1 = f1_score(
        valid_part["label"],
        predictions,
        labels=competition_labels,
        average="macro",
        zero_division=0,
    )
    return macro_f1, predictions


# 包含 epoch 0，避免微調變差後仍覆蓋原本較好的模型。
best_f1, best_predictions = evaluate_ft_model()
best_epoch = 0
best_state = {
    key: value.detach().cpu().clone()
    for key, value in ft_model.state_dict().items()
}

history = [{
    "epoch": 0,
    "train_loss": np.nan,
    "validation_macro_f1": best_f1,
}]

print(f"未微調，128 tokens：{best_f1:.4f}")

optimizer = torch.optim.AdamW(
    ft_model.parameters(),
    lr=5e-6,
    weight_decay=0.01,
)
# 新增這一段
counts = train_part["label"].value_counts()

class_weights = torch.ones(
    ft_model.config.num_labels,
    device=device,
)

for label in competition_labels:
    class_weights[ft_label_to_id[label]] = (
        len(train_part)
        / (len(competition_labels) * counts[label])
    ) ** 0.5

weighted_loss = torch.nn.CrossEntropyLoss(
    weight=class_weights
)

# 原本的迴圈接在下面

for epoch in range(1, EPOCHS + 1):
    ft_model.train()
    total_loss = 0.0
    total_samples = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for batch in progress:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        # 真正將 class weights 用於 loss。
        inputs = {
            key: value
            for key, value in batch.items()
            if key != "labels"
        }

        logits = ft_model(**inputs).logits
        loss = weighted_loss(logits, batch["labels"])

        if not torch.isfinite(loss).item():
            raise RuntimeError("Loss 出現 NaN 或 Inf，停止訓練。")
        
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            ft_model.parameters(), max_norm=1.0
        )
        optimizer.step()

        batch_count = batch["labels"].size(0)
        total_loss += loss.item() * batch_count
        total_samples += batch_count

        progress.set_postfix(loss=f"{loss.item():.4f}")

    val_f1, val_predictions = evaluate_ft_model()
    average_loss = total_loss / total_samples

    history.append({
        "epoch": epoch,
        "train_loss": average_loss,
        "validation_macro_f1": val_f1,
    })

    print(
        f"Epoch {epoch}: "
        f"loss={average_loss:.4f}, "
        f"validation macro F1={val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        best_predictions = val_predictions
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in ft_model.state_dict().items()
        }

# 恢復最佳權重，存到本機專案資料夾。
ft_model.load_state_dict(best_state)
ft_model.eval()

from datetime import datetime

save_dir = (
    "models/distilroberta_lr5e6_weighted_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
ft_model.save_pretrained(save_dir)
ft_tokenizer.save_pretrained(save_dir)

ft_history = pd.DataFrame(history)
display(ft_history.round(4))

print(f"最佳 epoch：{best_epoch}")
print(f"最佳 validation macro F1：{best_f1:.4f}")
print("模型儲存位置：", save_dir)

print(classification_report(
    valid_part["label"],
    best_predictions,
    labels=competition_labels,
    digits=4,
    zero_division=0,
))

未微調，128 tokens：0.5454


Epoch 1/3:   0%|          | 0/336 [00:00<?, ?it/s]

Epoch 1: loss=1.2474, validation macro F1=0.5668


Epoch 2/3:   0%|          | 0/336 [00:00<?, ?it/s]

Epoch 2: loss=1.0067, validation macro F1=0.5824


Epoch 3/3:   0%|          | 0/336 [00:00<?, ?it/s]

Epoch 3: loss=0.8915, validation macro F1=0.5821


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,epoch,train_loss,validation_macro_f1
0,0,NaN,0.5454
1,1,1.2474,0.5668
2,2,1.0067,0.5824
3,3,0.8915,0.5821


最佳 epoch：2
最佳 validation macro F1：0.5824
模型儲存位置： models/distilroberta_lr5e6_weighted_20261125_220354_790358
              precision    recall  f1-score   support

       anger     0.5414    0.5333    0.5373       135
     disgust     0.4588    0.4286    0.4432        91
        fear     0.5763    0.5862    0.5812        58
         joy     0.7485    0.7143    0.7310       175
     sadness     0.6204    0.6746    0.6464       126
    surprise     0.5435    0.5682    0.5556        88

    accuracy                         0.6018       673
   macro avg     0.5815    0.5842    0.5824       673
weighted avg     0.6022    0.6018    0.6015       673



In [41]:
# 記住剛才加權模型的獨立路徑。
weighted_model_dir = str(save_dir)

# 先讓新訓練模型回 CPU，騰出推論空間。
ft_model.cpu()
globals().pop("optimizer", None)
gc.collect()
torch.cuda.empty_cache()

# 這個既有函式使用全域變數，所以明確恢復驗證設定。
ensemble_labels = sorted(train["label"].unique())
valid_texts = (
    valid_part["text"].fillna("").astype(str).tolist()
)
valid_targets = valid_part["label"].to_numpy()

MAX_LENGTH = 128
BATCH_SIZE = 4
eval_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert set(train_part["clean"]).isdisjoint(valid_part["clean"])

# 全部都是只用 train_part 訓練的模型。
original_valid_probs = get_validation_probabilities(
    "models/distilroberta_lr5e6_unweighted"
)
weighted_valid_probs = get_validation_probabilities(
    weighted_model_dir
)
fixed_deberta_valid_probs = get_validation_probabilities(
    "models/deberta_v3_base_validation"
)

candidates = {
    "Original DistilRoBERTa": original_valid_probs,
    "Weighted DistilRoBERTa": weighted_valid_probs,
    "Original 50/50": (
        0.5 * original_valid_probs
        + 0.5 * fixed_deberta_valid_probs
    ),
    "Weighted 50/50": (
        0.5 * weighted_valid_probs
        + 0.5 * fixed_deberta_valid_probs
    ),
}

label_array = np.array(ensemble_labels)
comparison_rows = []

for name, probabilities in candidates.items():
    assert probabilities.shape == (
        len(valid_part), len(ensemble_labels)
    )
    assert np.isfinite(probabilities).all()

    predictions = label_array[probabilities.argmax(axis=1)]

    comparison_rows.append({
        "model": name,
        "validation_macro_f1": f1_score(
            valid_targets,
            predictions,
            labels=ensemble_labels,
            average="macro",
            zero_division=0,
        ),
    })

weighted_comparison = pd.DataFrame(comparison_rows)

display(
    weighted_comparison.sort_values(
        "validation_macro_f1", ascending=False
    ).round(4)
)

print("加權模型路徑：", weighted_model_dir)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

distilroberta_lr5e6_unweighted:   0%|          | 0/169 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

distilroberta_lr5e6_weighted_20261125_220354_790358:   0%|          | 0/169 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

deberta_v3_base_validation:   0%|          | 0/169 [00:00<?, ?it/s]

,model,validation_macro_f1
2,Original 50/50,0.5965
3,Weighted 50/50,0.5918
0,Original DistilRoBERTa,0.5913
1,Weighted DistilRoBERTa,0.5824


加權模型路徑： models/distilroberta_lr5e6_weighted_20261125_220354_790358


In [42]:
# 比例順序：Weighted DistilRoBERTa / DeBERTa
rows = []

for weight in [0.95, 0.50, 0.80]:
    probs = (
        weight * weighted_valid_probs
        + (1 - weight) * fixed_deberta_valid_probs
    )

    predictions = np.array(ensemble_labels)[
        probs.argmax(axis=1)
    ]

    rows.append({
        "Weighted Distil比例": weight,
        "DeBERTa比例": 1 - weight,
        "validation_macro_f1": f1_score(
            valid_targets,
            predictions,
            labels=ensemble_labels,
            average="macro",
            zero_division=0,
        ),
    })

display(pd.DataFrame(rows).round(4))
print("原本未加權 50/50 的參考分數：0.6098")

,Weighted Distil比例,DeBERTa比例,validation_macro_f1
0,0.95,0.05,0.5891
1,0.50,0.50,0.5918
2,0.80,0.20,0.5909


原本未加權 50/50 的參考分數：0.6098


## 最終5 fold 模型：Fold 2 (Final 5-fold model: Fold 2)



In [43]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# 與原本相同的分組方式，但取第 2 個 fold。
# Python 索引 1 = 第 2 個 fold。
fold2_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

fold2_splits = list(
    fold2_splitter.split(
        X=train["clean"],
        y=train["label"],
        groups=train["clean"],
    )
)

fold2_train_idx, fold2_valid_idx = fold2_splits[1]

fold2_train = train.iloc[fold2_train_idx].copy()
fold2_valid = train.iloc[fold2_valid_idx].copy()

assert set(fold2_train["clean"]).isdisjoint(
    set(fold2_valid["clean"])
)
assert set(fold2_train["post_id"]).isdisjoint(
    set(fold2_valid["post_id"])
)

print("第 2 個 fold 訓練筆數：", len(fold2_train))
print("第 2 個 fold 驗證筆數：", len(fold2_valid))

display(
    pd.concat(
        [
            fold2_train["label"].value_counts().rename("train"),
            fold2_valid["label"].value_counts().rename("valid"),
        ],
        axis=1,
    ).sort_index()
)

第 2 個 fold 訓練筆數： 2688
第 2 個 fold 驗證筆數： 672


,train,valid
label,,
anger,539,135
disgust,363,91
fear,230,58
joy,700,175
sadness,503,125
surprise,353,88


In [44]:
import gc
import random
from datetime import datetime
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "請先確認目前 Kernel 可以使用 CUDA。"

# 釋放前面實驗可能仍占用的 GPU 記憶體。
for fold2_old_name in ("ft_model", "deberta_model"):
    fold2_old_model = globals().get(fold2_old_name)
    if isinstance(fold2_old_model, torch.nn.Module):
        fold2_old_model.cpu()

gc.collect()
torch.cuda.empty_cache()

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

fold2_device = torch.device("cuda")
fold2_labels = sorted(train["label"].unique().tolist())

fold2_dir = Path("models") / (
    "distilroberta_fold2_unweighted_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
fold2_dir.mkdir(parents=True, exist_ok=False)

fold2_tokenizer = AutoTokenizer.from_pretrained(
    "j-hartmann/emotion-english-distilroberta-base"
)

fold2_model = AutoModelForSequenceClassification.from_pretrained(
    "j-hartmann/emotion-english-distilroberta-base"
).to(fold2_device)

# 保留pretrain model原本的七分類 head。
fold2_label_to_id = {
    str(label).lower(): int(index)
    for index, label in fold2_model.config.id2label.items()
}
assert set(fold2_labels).issubset(fold2_label_to_id)

fold2_selected_ids = [
    fold2_label_to_id[label] for label in fold2_labels
]


class Fold2EmotionDataset(Dataset):
    def __init__(self, frame):
        self.encodings = fold2_tokenizer(
            frame["text"].fillna("").astype(str).tolist(),
            truncation=True,
            max_length=128,
        )
        self.targets = [
            fold2_label_to_id[label]
            for label in frame["label"]
        ]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        item = {
            key: values[index]
            for key, values in self.encodings.items()
        }
        item["labels"] = self.targets[index]
        return item


fold2_collator = DataCollatorWithPadding(
    tokenizer=fold2_tokenizer
)

fold2_train_loader = DataLoader(
    Fold2EmotionDataset(fold2_train),
    batch_size=8,
    shuffle=True,
    collate_fn=fold2_collator,
    num_workers=0,
)

fold2_valid_loader = DataLoader(
    Fold2EmotionDataset(fold2_valid),
    batch_size=8,
    shuffle=False,
    collate_fn=fold2_collator,
    num_workers=0,
)


@torch.inference_mode()
def evaluate_fold2_distil():
    fold2_model.eval()
    probability_batches = []

    for batch in fold2_valid_loader:
        inputs = {
            key: value.to(fold2_device)
            for key, value in batch.items()
            if key != "labels"
        }

        logits = fold2_model(**inputs).logits

        # 只在競賽的六個類別之間計算機率。
        probabilities = torch.softmax(
            logits[:, fold2_selected_ids].float(),
            dim=1,
        )
        probability_batches.append(
            probabilities.cpu().numpy()
        )

    probabilities = np.concatenate(
        probability_batches, axis=0
    )
    predictions = np.asarray(fold2_labels)[
        probabilities.argmax(axis=1)
    ]

    score = f1_score(
        fold2_valid["label"].to_numpy(),
        predictions,
        labels=fold2_labels,
        average="macro",
        zero_division=0,
    )
    return score, probabilities


fold2_optimizer = torch.optim.AdamW(
    fold2_model.parameters(),
    lr=5e-6,
    weight_decay=0.01,
)

fold2_history = []
fold2_best_score = -1.0
fold2_best_epoch = None

for epoch in range(1, 3):
    fold2_model.train()
    loss_sum = 0.0
    sample_count = 0

    progress = tqdm(
        fold2_train_loader,
        desc=f"Fold 2 DistilRoBERTa epoch {epoch}/2",
    )

    for batch in progress:
        batch = {
            key: value.to(fold2_device)
            for key, value in batch.items()
        }

        fold2_optimizer.zero_grad(set_to_none=True)

        # 原版、不加權的交叉熵。
        loss = fold2_model(**batch).loss

        if not torch.isfinite(loss):
            raise RuntimeError("訓練 loss 非有限值，已停止。")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            fold2_model.parameters(), 1.0
        )
        fold2_optimizer.step()

        batch_size = batch["labels"].size(0)
        loss_sum += loss.item() * batch_size
        sample_count += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    score, probabilities = evaluate_fold2_distil()

    fold2_history.append({
        "epoch": epoch,
        "train_loss": loss_sum / sample_count,
        "validation_macro_f1": score,
    })

    print(
        f"Epoch {epoch}: "
        f"validation macro F1 = {score:.4f}"
    )

    if score > fold2_best_score:
        fold2_best_score = score
        fold2_best_epoch = epoch

        fold2_model.save_pretrained(fold2_dir)
        fold2_tokenizer.save_pretrained(fold2_dir)

        np.savez_compressed(
            fold2_dir / "validation_probabilities.npz",
            probabilities=probabilities,
            post_id=fold2_valid["post_id"].to_numpy(dtype=str),
            targets=fold2_valid["label"].to_numpy(dtype=str),
            labels=np.asarray(fold2_labels, dtype=str),
        )

pd.DataFrame(fold2_history).to_csv(
    fold2_dir / "training_history.csv",
    index=False,
)

# 儲存分組資料，後續讓 DeBERTa 使用完全相同的驗證集。
fold2_train[["post_id", "label"]].to_csv(
    fold2_dir / "train_rows.csv", index=False
)
fold2_valid[["post_id", "label"]].to_csv(
    fold2_dir / "validation_rows.csv", index=False
)

del fold2_optimizer
fold2_model.cpu()
gc.collect()
torch.cuda.empty_cache()

display(pd.DataFrame(fold2_history))
print("最佳 epoch：", fold2_best_epoch)
print("最佳 validation macro F1：", round(fold2_best_score, 4))
print("模型位置：", fold2_dir.resolve())

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Fold 2 DistilRoBERTa epoch 1/2:   0%|          | 0/336 [00:00<?, ?it/s]

Epoch 1: validation macro F1 = 0.6264


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fold 2 DistilRoBERTa epoch 2/2:   0%|          | 0/336 [00:00<?, ?it/s]

Epoch 2: validation macro F1 = 0.6384


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,epoch,train_loss,validation_macro_f1
0,1,1.250992,0.626401
1,2,1.013935,0.638419


最佳 epoch： 2
最佳 validation macro F1： 0.6384
模型位置： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\distilroberta_fold2_unweighted_20261125_220411_921220


In [ ]:
import math
from transformers import get_linear_schedule_with_warmup

# 前一格的 fold2_train、fold2_valid、fold2_dir 必須仍存在。
fold2_distil_dir = Path(fold2_dir)

fold2_db_dir = Path("models") / (
    "deberta_fold2_unweighted_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
fold2_db_dir.mkdir(parents=True, exist_ok=False)

gc.collect()
torch.cuda.empty_cache()

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

fold2_db_tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/deberta-v3-base",
    use_fast=False,
)

fold2_db_label_to_id = {
    label: index
    for index, label in enumerate(fold2_labels)
}

fold2_db_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=len(fold2_labels),
    label2id=fold2_db_label_to_id,
    id2label={
        index: label
        for label, index in fold2_db_label_to_id.items()
    },
).to(device=fold2_device, dtype=torch.float32)
assert all(
    parameter.dtype == torch.float32
    for parameter in fold2_db_model.parameters()
    if parameter.requires_grad
)

print("模型參數 dtype：", next(fold2_db_model.parameters()).dtype)


class Fold2DebertaDataset(Dataset):
    def __init__(self, frame):
        self.encodings = fold2_db_tokenizer(
            frame["text"].fillna("").astype(str).tolist(),
            truncation=True,
            max_length=128,
        )
        self.targets = [
            fold2_db_label_to_id[label]
            for label in frame["label"]
        ]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        item = {
            key: values[index]
            for key, values in self.encodings.items()
        }
        item["labels"] = self.targets[index]
        return item


fold2_db_collator = DataCollatorWithPadding(
    tokenizer=fold2_db_tokenizer,
    pad_to_multiple_of=8,
)

fold2_db_train_loader = DataLoader(
    Fold2DebertaDataset(fold2_train),
    batch_size=4,
    shuffle=True,
    collate_fn=fold2_db_collator,
    num_workers=0,
)

fold2_db_valid_loader = DataLoader(
    Fold2DebertaDataset(fold2_valid),
    batch_size=4,
    shuffle=False,
    collate_fn=fold2_db_collator,
    num_workers=0,
)


@torch.inference_mode()
def evaluate_fold2_deberta():
    fold2_db_model.eval()
    batches = []

    for batch in fold2_db_valid_loader:
        inputs = {
            key: value.to(fold2_device)
            for key, value in batch.items()
            if key != "labels"
        }

        # 與先前驗證一致，推論使用 FP32。
        logits = fold2_db_model(**inputs).logits
        batches.append(
            torch.softmax(logits.float(), dim=1).cpu().numpy()
        )

    probabilities = np.concatenate(batches, axis=0)
    predictions = np.asarray(fold2_labels)[
        probabilities.argmax(axis=1)
    ]

    score = f1_score(
        fold2_valid["label"].to_numpy(),
        predictions,
        labels=fold2_labels,
        average="macro",
        zero_division=0,
    )
    return score, probabilities


fold2_db_epochs = 4
fold2_db_accum = 4

fold2_db_optimizer = torch.optim.AdamW(
    fold2_db_model.parameters(),
    lr=2e-5,
    weight_decay=0.01,
)

fold2_db_total_updates = (
    math.ceil(len(fold2_db_train_loader) / fold2_db_accum)
    * fold2_db_epochs
)

fold2_db_scheduler = get_linear_schedule_with_warmup(
    fold2_db_optimizer,
    num_warmup_steps=int(fold2_db_total_updates * 0.1),
    num_training_steps=fold2_db_total_updates,
)

fold2_db_scaler = torch.amp.GradScaler("cuda")
fold2_db_history = []
fold2_db_best_score = -1.0
fold2_db_best_epoch = None

for epoch in range(1, fold2_db_epochs + 1):
    fold2_db_model.train()
    fold2_db_optimizer.zero_grad(set_to_none=True)

    loss_sum = 0.0
    sample_count = 0

    progress = tqdm(
        fold2_db_train_loader,
        desc=f"Fold 2 DeBERTa epoch {epoch}/{fold2_db_epochs}",
    )

    for step, batch in enumerate(progress):
        batch = {
            key: value.to(fold2_device)
            for key, value in batch.items()
        }

        # 最後不足 4 批的累積組，使用實際批數。
        group_start = (
            step // fold2_db_accum
        ) * fold2_db_accum
        group_size = min(
            fold2_db_accum,
            len(fold2_db_train_loader) - group_start,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            raw_loss = fold2_db_model(**batch).loss
            loss = raw_loss / group_size

        if not torch.isfinite(raw_loss):
            raise RuntimeError("訓練 loss 非有限值，已停止。")

        fold2_db_scaler.scale(loss).backward()

        should_update = (
            (step + 1) % fold2_db_accum == 0
            or step + 1 == len(fold2_db_train_loader)
        )

        if should_update:
            fold2_db_scaler.unscale_(fold2_db_optimizer)
            torch.nn.utils.clip_grad_norm_(
                fold2_db_model.parameters(), 1.0
            )

            previous_scale = fold2_db_scaler.get_scale()
            fold2_db_scaler.step(fold2_db_optimizer)
            fold2_db_scaler.update()

            if fold2_db_scaler.get_scale() >= previous_scale:
                fold2_db_scheduler.step()

            fold2_db_optimizer.zero_grad(set_to_none=True)

        batch_size = batch["labels"].size(0)
        loss_sum += raw_loss.item() * batch_size
        sample_count += batch_size
        progress.set_postfix(loss=f"{raw_loss.item():.4f}")

    score, probabilities = evaluate_fold2_deberta()

    fold2_db_history.append({
        "epoch": epoch,
        "train_loss": loss_sum / sample_count,
        "validation_macro_f1": score,
    })

    print(
        f"Epoch {epoch}: validation macro F1 = {score:.4f}"
    )

    if score > fold2_db_best_score:
        fold2_db_best_score = score
        fold2_db_best_epoch = epoch

        fold2_db_model.save_pretrained(fold2_db_dir)
        fold2_db_tokenizer.save_pretrained(fold2_db_dir)

        np.savez_compressed(
            fold2_db_dir / "validation_probabilities.npz",
            probabilities=probabilities,
            post_id=fold2_valid["post_id"].to_numpy(dtype=str),
            targets=fold2_valid["label"].to_numpy(dtype=str),
            labels=np.asarray(fold2_labels, dtype=str),
        )

pd.DataFrame(fold2_db_history).to_csv(
    fold2_db_dir / "training_history.csv",
    index=False,
)

del fold2_db_optimizer, fold2_db_scheduler, fold2_db_scaler
fold2_db_model.cpu()
gc.collect()
torch.cuda.empty_cache()

display(pd.DataFrame(fold2_db_history))
print("最佳 epoch：", fold2_db_best_epoch)
print("最佳 validation macro F1：", round(fold2_db_best_score, 4))
print("模型位置：", fold2_db_dir.resolve())

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.den

模型參數 dtype： torch.float32


Fold 2 DeBERTa epoch 1/4:   0%|          | 0/672 [00:00<?, ?it/s]

Epoch 1: validation macro F1 = 0.3556


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fold 2 DeBERTa epoch 2/4:   0%|          | 0/672 [00:00<?, ?it/s]

Epoch 2: validation macro F1 = 0.5185


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fold 2 DeBERTa epoch 3/4:   0%|          | 0/672 [00:00<?, ?it/s]

In [ ]:
with np.load(
    fold2_distil_dir / "validation_probabilities.npz",
    allow_pickle=True,
) as saved:
    fold2_distil_probs = saved["probabilities"].copy()
    fold2_distil_ids = saved["post_id"].copy()
    fold2_distil_labels = saved["labels"].copy()
    fold2_targets = saved["targets"].copy()

with np.load(
    fold2_db_dir / "validation_probabilities.npz",
    allow_pickle=True,
) as saved:
    fold2_deberta_probs = saved["probabilities"].copy()

    # 確認每一列資料與每一欄類別完全對齊。
    assert np.array_equal(fold2_distil_ids, saved["post_id"])
    assert np.array_equal(fold2_distil_labels, saved["labels"])
    assert np.array_equal(fold2_targets, saved["targets"])

fold2_comparison = []

for name, probabilities in {
    "Fold 2 DistilRoBERTa": fold2_distil_probs,
    "Fold 2 DeBERTa": fold2_deberta_probs,
    "Fold 2 Original 50/50": (
        0.5 * fold2_distil_probs
        + 0.5 * fold2_deberta_probs
    ),
}.items():
    predictions = fold2_distil_labels[
        probabilities.argmax(axis=1)
    ]

    fold2_comparison.append({
        "model": name,
        "validation_macro_f1": f1_score(
            fold2_targets,
            predictions,
            labels=fold2_distil_labels,
            average="macro",
            zero_division=0,
        ),
    })

display(
    pd.DataFrame(fold2_comparison).sort_values(
        "validation_macro_f1", ascending=False
    )
)

In [ ]:
fold2_ratio_results = []

for distil_weight in [0.0, 0.25, 0.5, 0.75, 1.0]:
    probabilities = (
        distil_weight * fold2_distil_probs
        + (1 - distil_weight) * fold2_deberta_probs
    )

    predictions = fold2_distil_labels[
        probabilities.argmax(axis=1)
    ]

    fold2_ratio_results.append({
        "DistilRoBERTa_weight": distil_weight,
        "DeBERTa_weight": 1 - distil_weight,
        "fold2_macro_f1": f1_score(
            fold2_targets,
            predictions,
            labels=fold2_distil_labels,
            average="macro",
            zero_division=0,
        ),
    })

ratio_comparison = pd.DataFrame(fold2_ratio_results)

# 第一個 fold 先前已實際測得的結果。
fold1_scores = {
    0.00: 0.5616,
    0.25: 0.5893,
    0.50: 0.6098,
    0.75: 0.5882,
    1.00: 0.5927,
}

ratio_comparison["fold1_macro_f1"] = (
    ratio_comparison["DistilRoBERTa_weight"].map(fold1_scores)
)

ratio_comparison["mean_macro_f1"] = (
    ratio_comparison["fold1_macro_f1"]
    + ratio_comparison["fold2_macro_f1"]
) / 2

display(
    ratio_comparison.sort_values(
        "mean_macro_f1", ascending=False
    ).reset_index(drop=True)
)

## 最終5 fold 模型：Fold 3 (Final 5-fold model: Fold 3)



In [ ]:
# 保存各 fold 的模型位置，避免下一輪變數被覆蓋後找不到。
if "cv_model_dirs" not in globals():
    cv_model_dirs = {}

cv_model_dirs[1] = {
    "distil": Path("models/distilroberta_lr5e6_unweighted"),
    "deberta": Path("models/deberta_v3_base_validation"),
}

cv_model_dirs[2] = {
    "distil": Path(fold2_distil_dir),
    "deberta": Path(fold2_db_dir),
}

for fold_number, paths in cv_model_dirs.items():
    for model_name, model_path in paths.items():
        assert (model_path / "config.json").exists(), (
            f"找不到 Fold {fold_number} {model_name}：{model_path}"
        )

# 同時存成檔案，重啟 Kernel 後仍能查到位置。
import json

with open("cv_model_dirs.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            str(number): {
                name: str(path)
                for name, path in paths.items()
            }
            for number, paths in cv_model_dirs.items()
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Fold 1、Fold 2 模型位置已保存。")

In [ ]:
CURRENT_FOLD = 3

current_train_idx, current_valid_idx = fold2_splits[
    CURRENT_FOLD - 1
]

fold2_train = train.iloc[current_train_idx].copy()
fold2_valid = train.iloc[current_valid_idx].copy()

assert set(fold2_train["clean"]).isdisjoint(
    set(fold2_valid["clean"])
)
assert set(fold2_train["post_id"]).isdisjoint(
    set(fold2_valid["post_id"])
)

print("目前實際訓練：Fold", CURRENT_FOLD)
print("訓練筆數：", len(fold2_train))
print("驗證筆數：", len(fold2_valid))

In [ ]:
import gc
import random
from datetime import datetime
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "請先確認目前 Kernel 可以使用 CUDA。"

# 釋放前面實驗可能仍占用的 GPU 記憶體。
for fold2_old_name in ("ft_model", "deberta_model"):
    fold2_old_model = globals().get(fold2_old_name)
    if isinstance(fold2_old_model, torch.nn.Module):
        fold2_old_model.cpu()

gc.collect()
torch.cuda.empty_cache()

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

fold2_device = torch.device("cuda")
fold2_labels = sorted(train["label"].unique().tolist())

fold2_dir = Path("models") / (
    "distilroberta_fold3_unweighted_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
fold2_dir.mkdir(parents=True, exist_ok=False)

fold2_tokenizer = AutoTokenizer.from_pretrained(
    "j-hartmann/emotion-english-distilroberta-base"
)

fold2_model = AutoModelForSequenceClassification.from_pretrained(
    "j-hartmann/emotion-english-distilroberta-base"
).to(fold2_device)

# 保留pretrain model原本的七分類 head。
fold2_label_to_id = {
    str(label).lower(): int(index)
    for index, label in fold2_model.config.id2label.items()
}
assert set(fold2_labels).issubset(fold2_label_to_id)

fold2_selected_ids = [
    fold2_label_to_id[label] for label in fold2_labels
]


class Fold2EmotionDataset(Dataset):
    def __init__(self, frame):
        self.encodings = fold2_tokenizer(
            frame["text"].fillna("").astype(str).tolist(),
            truncation=True,
            max_length=128,
        )
        self.targets = [
            fold2_label_to_id[label]
            for label in frame["label"]
        ]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        item = {
            key: values[index]
            for key, values in self.encodings.items()
        }
        item["labels"] = self.targets[index]
        return item


fold2_collator = DataCollatorWithPadding(
    tokenizer=fold2_tokenizer
)

fold2_train_loader = DataLoader(
    Fold2EmotionDataset(fold2_train),
    batch_size=8,
    shuffle=True,
    collate_fn=fold2_collator,
    num_workers=0,
)

fold2_valid_loader = DataLoader(
    Fold2EmotionDataset(fold2_valid),
    batch_size=8,
    shuffle=False,
    collate_fn=fold2_collator,
    num_workers=0,
)


@torch.inference_mode()
def evaluate_fold2_distil():
    fold2_model.eval()
    probability_batches = []

    for batch in fold2_valid_loader:
        inputs = {
            key: value.to(fold2_device)
            for key, value in batch.items()
            if key != "labels"
        }

        logits = fold2_model(**inputs).logits

        # 只在競賽的六個類別之間計算機率。
        probabilities = torch.softmax(
            logits[:, fold2_selected_ids].float(),
            dim=1,
        )
        probability_batches.append(
            probabilities.cpu().numpy()
        )

    probabilities = np.concatenate(
        probability_batches, axis=0
    )
    predictions = np.asarray(fold2_labels)[
        probabilities.argmax(axis=1)
    ]

    score = f1_score(
        fold2_valid["label"].to_numpy(),
        predictions,
        labels=fold2_labels,
        average="macro",
        zero_division=0,
    )
    return score, probabilities


fold2_optimizer = torch.optim.AdamW(
    fold2_model.parameters(),
    lr=5e-6,
    weight_decay=0.01,
)

fold2_history = []
fold2_best_score = -1.0
fold2_best_epoch = None

for epoch in range(1, 3):
    fold2_model.train()
    loss_sum = 0.0
    sample_count = 0

    progress = tqdm(
        fold2_train_loader,
        desc=f"Fold 2 DistilRoBERTa epoch {epoch}/2",
    )

    for batch in progress:
        batch = {
            key: value.to(fold2_device)
            for key, value in batch.items()
        }

        fold2_optimizer.zero_grad(set_to_none=True)

        # 原版、不加權的交叉熵。
        loss = fold2_model(**batch).loss

        if not torch.isfinite(loss):
            raise RuntimeError("訓練 loss 非有限值，已停止。")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            fold2_model.parameters(), 1.0
        )
        fold2_optimizer.step()

        batch_size = batch["labels"].size(0)
        loss_sum += loss.item() * batch_size
        sample_count += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    score, probabilities = evaluate_fold2_distil()

    fold2_history.append({
        "epoch": epoch,
        "train_loss": loss_sum / sample_count,
        "validation_macro_f1": score,
    })

    print(
        f"Epoch {epoch}: "
        f"validation macro F1 = {score:.4f}"
    )

    if score > fold2_best_score:
        fold2_best_score = score
        fold2_best_epoch = epoch

        fold2_model.save_pretrained(fold2_dir)
        fold2_tokenizer.save_pretrained(fold2_dir)

        np.savez_compressed(
            fold2_dir / "validation_probabilities.npz",
            probabilities=probabilities,
            post_id=fold2_valid["post_id"].to_numpy(dtype=str),
            targets=fold2_valid["label"].to_numpy(dtype=str),
            labels=np.asarray(fold2_labels, dtype=str),
        )

pd.DataFrame(fold2_history).to_csv(
    fold2_dir / "training_history.csv",
    index=False,
)

# 儲存分組資料，後續讓 DeBERTa 使用完全相同的驗證集。
fold2_train[["post_id", "label"]].to_csv(
    fold2_dir / "train_rows.csv", index=False
)
fold2_valid[["post_id", "label"]].to_csv(
    fold2_dir / "validation_rows.csv", index=False
)

del fold2_optimizer
fold2_model.cpu()
gc.collect()
torch.cuda.empty_cache()

display(pd.DataFrame(fold2_history))
print("最佳 epoch：", fold2_best_epoch)
print("最佳 validation macro F1：", round(fold2_best_score, 4))
print("模型位置：", fold2_dir.resolve())

In [51]:
import math
from transformers import get_linear_schedule_with_warmup

# 前一格的 fold2_train、fold2_valid、fold2_dir 必須仍存在。
fold2_distil_dir = Path(fold2_dir)

fold2_db_dir = Path("models") / (
    "deberta_fold3_unweighted_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
fold2_db_dir.mkdir(parents=True, exist_ok=False)

gc.collect()
torch.cuda.empty_cache()

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

fold2_db_tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/deberta-v3-base",
    use_fast=False,
)

fold2_db_label_to_id = {
    label: index
    for index, label in enumerate(fold2_labels)
}

fold2_db_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=len(fold2_labels),
    label2id=fold2_db_label_to_id,
    id2label={
        index: label
        for label, index in fold2_db_label_to_id.items()
    },
).to(device=fold2_device, dtype=torch.float32)
assert all(
    parameter.dtype == torch.float32
    for parameter in fold2_db_model.parameters()
    if parameter.requires_grad
)

print("模型參數 dtype：", next(fold2_db_model.parameters()).dtype)


class Fold2DebertaDataset(Dataset):
    def __init__(self, frame):
        self.encodings = fold2_db_tokenizer(
            frame["text"].fillna("").astype(str).tolist(),
            truncation=True,
            max_length=128,
        )
        self.targets = [
            fold2_db_label_to_id[label]
            for label in frame["label"]
        ]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        item = {
            key: values[index]
            for key, values in self.encodings.items()
        }
        item["labels"] = self.targets[index]
        return item


fold2_db_collator = DataCollatorWithPadding(
    tokenizer=fold2_db_tokenizer,
    pad_to_multiple_of=8,
)

fold2_db_train_loader = DataLoader(
    Fold2DebertaDataset(fold2_train),
    batch_size=4,
    shuffle=True,
    collate_fn=fold2_db_collator,
    num_workers=0,
)

fold2_db_valid_loader = DataLoader(
    Fold2DebertaDataset(fold2_valid),
    batch_size=4,
    shuffle=False,
    collate_fn=fold2_db_collator,
    num_workers=0,
)


@torch.inference_mode()
def evaluate_fold2_deberta():
    fold2_db_model.eval()
    batches = []

    for batch in fold2_db_valid_loader:
        inputs = {
            key: value.to(fold2_device)
            for key, value in batch.items()
            if key != "labels"
        }

        # 與先前驗證一致，推論使用 FP32。
        logits = fold2_db_model(**inputs).logits
        batches.append(
            torch.softmax(logits.float(), dim=1).cpu().numpy()
        )

    probabilities = np.concatenate(batches, axis=0)
    predictions = np.asarray(fold2_labels)[
        probabilities.argmax(axis=1)
    ]

    score = f1_score(
        fold2_valid["label"].to_numpy(),
        predictions,
        labels=fold2_labels,
        average="macro",
        zero_division=0,
    )
    return score, probabilities


fold2_db_epochs = 4
fold2_db_accum = 4

fold2_db_optimizer = torch.optim.AdamW(
    fold2_db_model.parameters(),
    lr=2e-5,
    weight_decay=0.01,
)

fold2_db_total_updates = (
    math.ceil(len(fold2_db_train_loader) / fold2_db_accum)
    * fold2_db_epochs
)

fold2_db_scheduler = get_linear_schedule_with_warmup(
    fold2_db_optimizer,
    num_warmup_steps=int(fold2_db_total_updates * 0.1),
    num_training_steps=fold2_db_total_updates,
)

fold2_db_scaler = torch.amp.GradScaler("cuda")
fold2_db_history = []
fold2_db_best_score = -1.0
fold2_db_best_epoch = None

for epoch in range(1, fold2_db_epochs + 1):
    fold2_db_model.train()
    fold2_db_optimizer.zero_grad(set_to_none=True)

    loss_sum = 0.0
    sample_count = 0

    progress = tqdm(
        fold2_db_train_loader,
        desc=f"Fold 2 DeBERTa epoch {epoch}/{fold2_db_epochs}",
    )

    for step, batch in enumerate(progress):
        batch = {
            key: value.to(fold2_device)
            for key, value in batch.items()
        }

        # 最後不足 4 批的累積組，使用實際批數。
        group_start = (
            step // fold2_db_accum
        ) * fold2_db_accum
        group_size = min(
            fold2_db_accum,
            len(fold2_db_train_loader) - group_start,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            raw_loss = fold2_db_model(**batch).loss
            loss = raw_loss / group_size

        if not torch.isfinite(raw_loss):
            raise RuntimeError("訓練 loss 非有限值，已停止。")

        fold2_db_scaler.scale(loss).backward()

        should_update = (
            (step + 1) % fold2_db_accum == 0
            or step + 1 == len(fold2_db_train_loader)
        )

        if should_update:
            fold2_db_scaler.unscale_(fold2_db_optimizer)
            torch.nn.utils.clip_grad_norm_(
                fold2_db_model.parameters(), 1.0
            )

            previous_scale = fold2_db_scaler.get_scale()
            fold2_db_scaler.step(fold2_db_optimizer)
            fold2_db_scaler.update()

            if fold2_db_scaler.get_scale() >= previous_scale:
                fold2_db_scheduler.step()

            fold2_db_optimizer.zero_grad(set_to_none=True)

        batch_size = batch["labels"].size(0)
        loss_sum += raw_loss.item() * batch_size
        sample_count += batch_size
        progress.set_postfix(loss=f"{raw_loss.item():.4f}")

    score, probabilities = evaluate_fold2_deberta()

    fold2_db_history.append({
        "epoch": epoch,
        "train_loss": loss_sum / sample_count,
        "validation_macro_f1": score,
    })

    print(
        f"Epoch {epoch}: validation macro F1 = {score:.4f}"
    )

    if score > fold2_db_best_score:
        fold2_db_best_score = score
        fold2_db_best_epoch = epoch

        fold2_db_model.save_pretrained(fold2_db_dir)
        fold2_db_tokenizer.save_pretrained(fold2_db_dir)

        np.savez_compressed(
            fold2_db_dir / "validation_probabilities.npz",
            probabilities=probabilities,
            post_id=fold2_valid["post_id"].to_numpy(dtype=str),
            targets=fold2_valid["label"].to_numpy(dtype=str),
            labels=np.asarray(fold2_labels, dtype=str),
        )

pd.DataFrame(fold2_db_history).to_csv(
    fold2_db_dir / "training_history.csv",
    index=False,
)

del fold2_db_optimizer, fold2_db_scheduler, fold2_db_scaler
fold2_db_model.cpu()
gc.collect()
torch.cuda.empty_cache()

display(pd.DataFrame(fold2_db_history))
print("最佳 epoch：", fold2_db_best_epoch)
print("最佳 validation macro F1：", round(fold2_db_best_score, 4))
print("模型位置：", fold2_db_dir.resolve())

Epoch 3: validation macro F1 = 0.5837


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fold 2 DeBERTa epoch 4/4:   0%|          | 0/672 [00:00<?, ?it/s]

Epoch 4: validation macro F1 = 0.6123


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,epoch,train_loss,validation_macro_f1
0,1,1.658346,0.306994
1,2,1.251118,0.512268
2,3,0.934717,0.583743
3,4,0.729250,0.612279


最佳 epoch： 4
最佳 validation macro F1： 0.6123
模型位置： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\deberta_fold3_unweighted_20261125_221150_886276


In [52]:
fold3_distil_dir = Path(fold2_distil_dir)
fold3_deberta_dir = Path(fold2_db_dir)
assert "distilroberta_fold3_" in fold3_distil_dir.name
assert "deberta_fold3_" in fold3_deberta_dir.name
# 下一格直接由 checkpoint 重算驗證機率。


In [53]:
import gc
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

# 直接由分組索引重建 Fold 3 驗證集。
repair_valid_idx = fold2_splits[2][1]
repair_valid = train.iloc[repair_valid_idx].copy()

repair_labels = np.asarray(
    sorted(train["label"].unique().tolist()),
    dtype=str,
)
repair_targets = repair_valid["label"].to_numpy(dtype=str)
repair_ids = repair_valid["post_id"].to_numpy(dtype=str)
repair_texts = (
    repair_valid["text"].fillna("").astype(str).tolist()
)
repair_device = torch.device("cuda")

assert torch.cuda.is_available()

# 釋放前面模型可能占用的 GPU 記憶體。
for name in ("ft_model", "deberta_model",
             "fold2_model", "fold2_db_model"):
    old_model = globals().get(name)
    if isinstance(old_model, torch.nn.Module):
        old_model.cpu()

gc.collect()
torch.cuda.empty_cache()


@torch.inference_mode()
def repair_predict(model_dir, expected_type):
    tokenizer = AutoTokenizer.from_pretrained(
        model_dir,
        use_fast=False,
        local_files_only=True,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_dir,
        local_files_only=True,
    )

    assert model.config.model_type == expected_type

    model = model.to(repair_device).eval()

    try:
        label_to_id = {
            str(label).lower(): int(index)
            for index, label in model.config.id2label.items()
        }
        selected_ids = [
            label_to_id[label] for label in repair_labels
        ]

        batches = []

        for start in tqdm(
            range(0, len(repair_texts), 4),
            desc=f"重新推論 {expected_type}",
        ):
            inputs = tokenizer(
                repair_texts[start:start + 4],
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt",
            ).to(repair_device)

            logits = model(**inputs).logits
            probabilities = torch.softmax(
                logits[:, selected_ids].float(),
                dim=1,
            )
            batches.append(probabilities.cpu().numpy())

        return np.concatenate(batches, axis=0)

    finally:
        model.cpu()
        del model
        gc.collect()
        torch.cuda.empty_cache()


# 分別從兩個模型重新計算。
fold3_distil_probs = repair_predict(
    fold3_distil_dir, "roberta"
)
fold3_deberta_probs = repair_predict(
    fold3_deberta_dir, "deberta-v2"
)

assert not np.array_equal(
    fold3_distil_probs, fold3_deberta_probs
), "重新推論仍完全相同，請停止並檢查模型路徑。"

# 另外保存修復版；字串明確轉為 str，避免 object 讀取問題。
for directory, probabilities in [
    (fold3_distil_dir, fold3_distil_probs),
    (fold3_deberta_dir, fold3_deberta_probs),
]:
    np.savez_compressed(
        directory / "validation_probabilities_repaired.npz",
        probabilities=probabilities,
        post_id=repair_ids,
        targets=repair_targets,
        labels=repair_labels,
    )

repair_results = []

for name, probabilities in {
    "Fold 3 DistilRoBERTa": fold3_distil_probs,
    "Fold 3 DeBERTa": fold3_deberta_probs,
    "Fold 3 Original 50/50": (
        0.5 * fold3_distil_probs
        + 0.5 * fold3_deberta_probs
    ),
}.items():
    predictions = repair_labels[
        probabilities.argmax(axis=1)
    ]
    repair_results.append({
        "model": name,
        "validation_macro_f1": f1_score(
            repair_targets,
            predictions,
            labels=repair_labels,
            average="macro",
            zero_division=0,
        ),
    })

display(pd.DataFrame(repair_results))
# 推論完成後登記本 fold，避免缺少 Fold 3。
cv_model_dirs[3] = {"distil": fold3_distil_dir, "deberta": fold3_deberta_dir}
with open("cv_model_dirs.json", "w", encoding="utf-8") as file:
    json.dump({str(k): {name: str(path) for name, path in v.items()}
               for k, v in cv_model_dirs.items()}, file, indent=2)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

重新推論 roberta:   0%|          | 0/168 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

重新推論 deberta-v2:   0%|          | 0/168 [00:00<?, ?it/s]

,model,validation_macro_f1
0,Fold 3 DistilRoBERTa,0.615704
1,Fold 3 DeBERTa,0.612279
2,Fold 3 Original 50/50,0.636828


## 最終5 fold 模型：Fold 4 (Final 5-fold model: Fold 4)



In [54]:
CURRENT_FOLD = 4
current_train_idx, current_valid_idx = fold2_splits[
    CURRENT_FOLD - 1
]

# 沿用既有訓練程式的變數名稱。
fold2_train = train.iloc[current_train_idx].copy()
fold2_valid = train.iloc[current_valid_idx].copy()

assert set(fold2_train["clean"]).isdisjoint(
    set(fold2_valid["clean"])
)

print("目前實際訓練：Fold", CURRENT_FOLD)
print("訓練筆數：", len(fold2_train))
print("驗證筆數：", len(fold2_valid))

目前實際訓練：Fold 4
訓練筆數： 2688
驗證筆數： 672


In [55]:
import gc
import random
from datetime import datetime
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "請先確認目前 Kernel 可以使用 CUDA。"

# 釋放前面實驗可能仍占用的 GPU 記憶體。
for fold2_old_name in ("ft_model", "deberta_model"):
    fold2_old_model = globals().get(fold2_old_name)
    if isinstance(fold2_old_model, torch.nn.Module):
        fold2_old_model.cpu()

gc.collect()
torch.cuda.empty_cache()

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

fold2_device = torch.device("cuda")
fold2_labels = sorted(train["label"].unique().tolist())

fold2_dir = Path("models") / (
    "distilroberta_fold4_unweighted_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
fold2_dir.mkdir(parents=True, exist_ok=False)

fold2_tokenizer = AutoTokenizer.from_pretrained(
    "j-hartmann/emotion-english-distilroberta-base"
)

fold2_model = AutoModelForSequenceClassification.from_pretrained(
    "j-hartmann/emotion-english-distilroberta-base"
).to(fold2_device)

# 保留pretrain model原本的七分類 head。
fold2_label_to_id = {
    str(label).lower(): int(index)
    for index, label in fold2_model.config.id2label.items()
}
assert set(fold2_labels).issubset(fold2_label_to_id)

fold2_selected_ids = [
    fold2_label_to_id[label] for label in fold2_labels
]


class Fold2EmotionDataset(Dataset):
    def __init__(self, frame):
        self.encodings = fold2_tokenizer(
            frame["text"].fillna("").astype(str).tolist(),
            truncation=True,
            max_length=128,
        )
        self.targets = [
            fold2_label_to_id[label]
            for label in frame["label"]
        ]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        item = {
            key: values[index]
            for key, values in self.encodings.items()
        }
        item["labels"] = self.targets[index]
        return item


fold2_collator = DataCollatorWithPadding(
    tokenizer=fold2_tokenizer
)

fold2_train_loader = DataLoader(
    Fold2EmotionDataset(fold2_train),
    batch_size=8,
    shuffle=True,
    collate_fn=fold2_collator,
    num_workers=0,
)

fold2_valid_loader = DataLoader(
    Fold2EmotionDataset(fold2_valid),
    batch_size=8,
    shuffle=False,
    collate_fn=fold2_collator,
    num_workers=0,
)


@torch.inference_mode()
def evaluate_fold2_distil():
    fold2_model.eval()
    probability_batches = []

    for batch in fold2_valid_loader:
        inputs = {
            key: value.to(fold2_device)
            for key, value in batch.items()
            if key != "labels"
        }

        logits = fold2_model(**inputs).logits

        # 只在競賽的六個類別之間計算機率。
        probabilities = torch.softmax(
            logits[:, fold2_selected_ids].float(),
            dim=1,
        )
        probability_batches.append(
            probabilities.cpu().numpy()
        )

    probabilities = np.concatenate(
        probability_batches, axis=0
    )
    predictions = np.asarray(fold2_labels)[
        probabilities.argmax(axis=1)
    ]

    score = f1_score(
        fold2_valid["label"].to_numpy(),
        predictions,
        labels=fold2_labels,
        average="macro",
        zero_division=0,
    )
    return score, probabilities


fold2_optimizer = torch.optim.AdamW(
    fold2_model.parameters(),
    lr=5e-6,
    weight_decay=0.01,
)

fold2_history = []
fold2_best_score = -1.0
fold2_best_epoch = None

for epoch in range(1, 3):
    fold2_model.train()
    loss_sum = 0.0
    sample_count = 0

    progress = tqdm(
        fold2_train_loader,
        desc=f"Fold 2 DistilRoBERTa epoch {epoch}/2",
    )

    for batch in progress:
        batch = {
            key: value.to(fold2_device)
            for key, value in batch.items()
        }

        fold2_optimizer.zero_grad(set_to_none=True)

        # 原版、不加權的交叉熵。
        loss = fold2_model(**batch).loss

        if not torch.isfinite(loss):
            raise RuntimeError("訓練 loss 非有限值，已停止。")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            fold2_model.parameters(), 1.0
        )
        fold2_optimizer.step()

        batch_size = batch["labels"].size(0)
        loss_sum += loss.item() * batch_size
        sample_count += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    score, probabilities = evaluate_fold2_distil()

    fold2_history.append({
        "epoch": epoch,
        "train_loss": loss_sum / sample_count,
        "validation_macro_f1": score,
    })

    print(
        f"Epoch {epoch}: "
        f"validation macro F1 = {score:.4f}"
    )

    if score > fold2_best_score:
        fold2_best_score = score
        fold2_best_epoch = epoch

        fold2_model.save_pretrained(fold2_dir)
        fold2_tokenizer.save_pretrained(fold2_dir)

        np.savez_compressed(
            fold2_dir / "validation_probabilities.npz",
            probabilities=probabilities,
            post_id=fold2_valid["post_id"].to_numpy(dtype=str),
            targets=fold2_valid["label"].to_numpy(dtype=str),
            labels=np.asarray(fold2_labels, dtype=str),
        )

pd.DataFrame(fold2_history).to_csv(
    fold2_dir / "training_history.csv",
    index=False,
)

# 儲存分組資料，後續讓 DeBERTa 使用完全相同的驗證集。
fold2_train[["post_id", "label"]].to_csv(
    fold2_dir / "train_rows.csv", index=False
)
fold2_valid[["post_id", "label"]].to_csv(
    fold2_dir / "validation_rows.csv", index=False
)

del fold2_optimizer
fold2_model.cpu()
gc.collect()
torch.cuda.empty_cache()

display(pd.DataFrame(fold2_history))
print("最佳 epoch：", fold2_best_epoch)
print("最佳 validation macro F1：", round(fold2_best_score, 4))
print("模型位置：", fold2_dir.resolve())

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Fold 2 DistilRoBERTa epoch 1/2:   0%|          | 0/336 [00:00<?, ?it/s]

Epoch 1: validation macro F1 = 0.6328


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fold 2 DistilRoBERTa epoch 2/2:   0%|          | 0/336 [00:00<?, ?it/s]

Epoch 2: validation macro F1 = 0.6299


,epoch,train_loss,validation_macro_f1
0,1,1.253841,0.632793
1,2,1.008589,0.629866


最佳 epoch： 1
最佳 validation macro F1： 0.6328
模型位置： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\distilroberta_fold4_unweighted_20261125_221810_369695


In [56]:
import math
from transformers import get_linear_schedule_with_warmup

# 前一格的 fold2_train、fold2_valid、fold2_dir 必須仍存在。
fold2_distil_dir = Path(fold2_dir)

fold2_db_dir = Path("models") / (
    "deberta_fold4_unweighted_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
fold2_db_dir.mkdir(parents=True, exist_ok=False)

gc.collect()
torch.cuda.empty_cache()

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

fold2_db_tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/deberta-v3-base",
    use_fast=False,
)

fold2_db_label_to_id = {
    label: index
    for index, label in enumerate(fold2_labels)
}

fold2_db_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=len(fold2_labels),
    label2id=fold2_db_label_to_id,
    id2label={
        index: label
        for label, index in fold2_db_label_to_id.items()
    },
).to(device=fold2_device, dtype=torch.float32)
assert all(
    parameter.dtype == torch.float32
    for parameter in fold2_db_model.parameters()
    if parameter.requires_grad
)

print("模型參數 dtype：", next(fold2_db_model.parameters()).dtype)


class Fold2DebertaDataset(Dataset):
    def __init__(self, frame):
        self.encodings = fold2_db_tokenizer(
            frame["text"].fillna("").astype(str).tolist(),
            truncation=True,
            max_length=128,
        )
        self.targets = [
            fold2_db_label_to_id[label]
            for label in frame["label"]
        ]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        item = {
            key: values[index]
            for key, values in self.encodings.items()
        }
        item["labels"] = self.targets[index]
        return item


fold2_db_collator = DataCollatorWithPadding(
    tokenizer=fold2_db_tokenizer,
    pad_to_multiple_of=8,
)

fold2_db_train_loader = DataLoader(
    Fold2DebertaDataset(fold2_train),
    batch_size=4,
    shuffle=True,
    collate_fn=fold2_db_collator,
    num_workers=0,
)

fold2_db_valid_loader = DataLoader(
    Fold2DebertaDataset(fold2_valid),
    batch_size=4,
    shuffle=False,
    collate_fn=fold2_db_collator,
    num_workers=0,
)


@torch.inference_mode()
def evaluate_fold2_deberta():
    fold2_db_model.eval()
    batches = []

    for batch in fold2_db_valid_loader:
        inputs = {
            key: value.to(fold2_device)
            for key, value in batch.items()
            if key != "labels"
        }

        # 與先前驗證一致，推論使用 FP32。
        logits = fold2_db_model(**inputs).logits
        batches.append(
            torch.softmax(logits.float(), dim=1).cpu().numpy()
        )

    probabilities = np.concatenate(batches, axis=0)
    predictions = np.asarray(fold2_labels)[
        probabilities.argmax(axis=1)
    ]

    score = f1_score(
        fold2_valid["label"].to_numpy(),
        predictions,
        labels=fold2_labels,
        average="macro",
        zero_division=0,
    )
    return score, probabilities


fold2_db_epochs = 4
fold2_db_accum = 4

fold2_db_optimizer = torch.optim.AdamW(
    fold2_db_model.parameters(),
    lr=2e-5,
    weight_decay=0.01,
)

fold2_db_total_updates = (
    math.ceil(len(fold2_db_train_loader) / fold2_db_accum)
    * fold2_db_epochs
)

fold2_db_scheduler = get_linear_schedule_with_warmup(
    fold2_db_optimizer,
    num_warmup_steps=int(fold2_db_total_updates * 0.1),
    num_training_steps=fold2_db_total_updates,
)

fold2_db_scaler = torch.amp.GradScaler("cuda")
fold2_db_history = []
fold2_db_best_score = -1.0
fold2_db_best_epoch = None

for epoch in range(1, fold2_db_epochs + 1):
    fold2_db_model.train()
    fold2_db_optimizer.zero_grad(set_to_none=True)

    loss_sum = 0.0
    sample_count = 0

    progress = tqdm(
        fold2_db_train_loader,
        desc=f"Fold 2 DeBERTa epoch {epoch}/{fold2_db_epochs}",
    )

    for step, batch in enumerate(progress):
        batch = {
            key: value.to(fold2_device)
            for key, value in batch.items()
        }

        # 最後不足 4 批的累積組，使用實際批數。
        group_start = (
            step // fold2_db_accum
        ) * fold2_db_accum
        group_size = min(
            fold2_db_accum,
            len(fold2_db_train_loader) - group_start,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            raw_loss = fold2_db_model(**batch).loss
            loss = raw_loss / group_size

        if not torch.isfinite(raw_loss):
            raise RuntimeError("訓練 loss 非有限值，已停止。")

        fold2_db_scaler.scale(loss).backward()

        should_update = (
            (step + 1) % fold2_db_accum == 0
            or step + 1 == len(fold2_db_train_loader)
        )

        if should_update:
            fold2_db_scaler.unscale_(fold2_db_optimizer)
            torch.nn.utils.clip_grad_norm_(
                fold2_db_model.parameters(), 1.0
            )

            previous_scale = fold2_db_scaler.get_scale()
            fold2_db_scaler.step(fold2_db_optimizer)
            fold2_db_scaler.update()

            if fold2_db_scaler.get_scale() >= previous_scale:
                fold2_db_scheduler.step()

            fold2_db_optimizer.zero_grad(set_to_none=True)

        batch_size = batch["labels"].size(0)
        loss_sum += raw_loss.item() * batch_size
        sample_count += batch_size
        progress.set_postfix(loss=f"{raw_loss.item():.4f}")

    score, probabilities = evaluate_fold2_deberta()

    fold2_db_history.append({
        "epoch": epoch,
        "train_loss": loss_sum / sample_count,
        "validation_macro_f1": score,
    })

    print(
        f"Epoch {epoch}: validation macro F1 = {score:.4f}"
    )

    if score > fold2_db_best_score:
        fold2_db_best_score = score
        fold2_db_best_epoch = epoch

        fold2_db_model.save_pretrained(fold2_db_dir)
        fold2_db_tokenizer.save_pretrained(fold2_db_dir)

        np.savez_compressed(
            fold2_db_dir / "validation_probabilities.npz",
            probabilities=probabilities,
            post_id=fold2_valid["post_id"].to_numpy(dtype=str),
            targets=fold2_valid["label"].to_numpy(dtype=str),
            labels=np.asarray(fold2_labels, dtype=str),
        )

pd.DataFrame(fold2_db_history).to_csv(
    fold2_db_dir / "training_history.csv",
    index=False,
)

del fold2_db_optimizer, fold2_db_scheduler, fold2_db_scaler
fold2_db_model.cpu()
gc.collect()
torch.cuda.empty_cache()

display(pd.DataFrame(fold2_db_history))
print("最佳 epoch：", fold2_db_best_epoch)
print("最佳 validation macro F1：", round(fold2_db_best_score, 4))
print("模型位置：", fold2_db_dir.resolve())

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.den

模型參數 dtype： torch.float32


Fold 2 DeBERTa epoch 1/4:   0%|          | 0/672 [00:00<?, ?it/s]

Epoch 1: validation macro F1 = 0.2484


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (7 outputs left out)

Epoch 4: validation macro F1 = 0.5850


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,epoch,train_loss,validation_macro_f1
0,1,1.676622,0.248401
1,2,1.216048,0.485532
2,3,0.922598,0.560897
3,4,0.731194,0.585013


最佳 epoch： 4
最佳 validation macro F1： 0.585
模型位置： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\deberta_fold4_unweighted_20261125_221845_562861


In [57]:
fold4_distil_dir = Path(fold2_distil_dir)
fold4_deberta_dir = Path(fold2_db_dir)

for directory, expected_name, expected_type in [
    (fold4_distil_dir, "distilroberta_fold4_", "roberta"),
    (fold4_deberta_dir, "deberta_fold4_", "deberta-v2"),
]:
    assert expected_name in directory.name, str(directory)

    with open(directory / "config.json", encoding="utf-8") as file:
        config = json.load(file)

    assert config["model_type"] == expected_type

cv_model_dirs[4] = {
    "distil": fold4_distil_dir,
    "deberta": fold4_deberta_dir,
}

with open("cv_model_dirs.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            str(number): {
                name: str(path)
                for name, path in paths.items()
            }
            for number, paths in cv_model_dirs.items()
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Fold 4 模型位置已確認並保存。")

Fold 4 模型位置已確認並保存。


In [58]:
repair_valid = fold2_valid.copy()
repair_labels = np.asarray(
    sorted(train["label"].unique().tolist()), dtype=str
)
repair_targets = repair_valid["label"].to_numpy(dtype=str)
repair_ids = repair_valid["post_id"].to_numpy(dtype=str)
repair_texts = (
    repair_valid["text"].fillna("").astype(str).tolist()
)

fold4_distil_probs = repair_predict(
    fold4_distil_dir, "roberta"
)
fold4_deberta_probs = repair_predict(
    fold4_deberta_dir, "deberta-v2"
)

fold4_results = []

for name, probabilities in {
    "Fold 4 DistilRoBERTa": fold4_distil_probs,
    "Fold 4 DeBERTa": fold4_deberta_probs,
    "Fold 4 Original 50/50": (
        0.5 * fold4_distil_probs
        + 0.5 * fold4_deberta_probs
    ),
}.items():
    fold4_results.append({
        "model": name,
        "validation_macro_f1": f1_score(
            repair_targets,
            repair_labels[probabilities.argmax(axis=1)],
            labels=repair_labels,
            average="macro",
            zero_division=0,
        ),
    })

for directory, probabilities in [
    (fold4_distil_dir, fold4_distil_probs),
    (fold4_deberta_dir, fold4_deberta_probs),
]:
    np.savez_compressed(
        directory / "validation_probabilities_repaired.npz",
        probabilities=probabilities,
        post_id=repair_ids,
        targets=repair_targets,
        labels=repair_labels,
    )

display(pd.DataFrame(fold4_results))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

重新推論 roberta:   0%|          | 0/168 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

重新推論 deberta-v2:   0%|          | 0/168 [00:00<?, ?it/s]

,model,validation_macro_f1
0,Fold 4 DistilRoBERTa,0.632793
1,Fold 4 DeBERTa,0.585013
2,Fold 4 Original 50/50,0.619718


## 最終5 fold 模型：Fold 5 (Final 5-fold model: Fold 5)


In [59]:
CURRENT_FOLD = 5

current_train_idx, current_valid_idx = fold2_splits[4]

fold2_train = train.iloc[current_train_idx].copy()
fold2_valid = train.iloc[current_valid_idx].copy()

assert set(fold2_train["clean"]).isdisjoint(
    set(fold2_valid["clean"])
)

print("目前實際訓練：Fold", CURRENT_FOLD)
print("訓練筆數：", len(fold2_train))
print("驗證筆數：", len(fold2_valid))

目前實際訓練：Fold 5
訓練筆數： 2689
驗證筆數： 671


In [60]:
## Fold 5
import gc
import random
from datetime import datetime
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "請先確認目前 Kernel 可以使用 CUDA。"

# 釋放前面實驗可能仍占用的 GPU 記憶體。
for fold2_old_name in ("ft_model", "deberta_model"):
    fold2_old_model = globals().get(fold2_old_name)
    if isinstance(fold2_old_model, torch.nn.Module):
        fold2_old_model.cpu()

gc.collect()
torch.cuda.empty_cache()

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

fold2_device = torch.device("cuda")
fold2_labels = sorted(train["label"].unique().tolist())

fold2_dir = Path("models") / (
    "distilroberta_fold5_unweighted_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
fold2_dir.mkdir(parents=True, exist_ok=False)

fold2_tokenizer = AutoTokenizer.from_pretrained(
    "j-hartmann/emotion-english-distilroberta-base"
)

fold2_model = AutoModelForSequenceClassification.from_pretrained(
    "j-hartmann/emotion-english-distilroberta-base"
).to(fold2_device)

# 保留pretrain model原本的七分類 head。
fold2_label_to_id = {
    str(label).lower(): int(index)
    for index, label in fold2_model.config.id2label.items()
}
assert set(fold2_labels).issubset(fold2_label_to_id)

fold2_selected_ids = [
    fold2_label_to_id[label] for label in fold2_labels
]


class Fold2EmotionDataset(Dataset):
    def __init__(self, frame):
        self.encodings = fold2_tokenizer(
            frame["text"].fillna("").astype(str).tolist(),
            truncation=True,
            max_length=128,
        )
        self.targets = [
            fold2_label_to_id[label]
            for label in frame["label"]
        ]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        item = {
            key: values[index]
            for key, values in self.encodings.items()
        }
        item["labels"] = self.targets[index]
        return item


fold2_collator = DataCollatorWithPadding(
    tokenizer=fold2_tokenizer
)

fold2_train_loader = DataLoader(
    Fold2EmotionDataset(fold2_train),
    batch_size=8,
    shuffle=True,
    collate_fn=fold2_collator,
    num_workers=0,
)

fold2_valid_loader = DataLoader(
    Fold2EmotionDataset(fold2_valid),
    batch_size=8,
    shuffle=False,
    collate_fn=fold2_collator,
    num_workers=0,
)


@torch.inference_mode()
def evaluate_fold2_distil():
    fold2_model.eval()
    probability_batches = []

    for batch in fold2_valid_loader:
        inputs = {
            key: value.to(fold2_device)
            for key, value in batch.items()
            if key != "labels"
        }

        logits = fold2_model(**inputs).logits

        # 只在競賽的六個類別之間計算機率。
        probabilities = torch.softmax(
            logits[:, fold2_selected_ids].float(),
            dim=1,
        )
        probability_batches.append(
            probabilities.cpu().numpy()
        )

    probabilities = np.concatenate(
        probability_batches, axis=0
    )
    predictions = np.asarray(fold2_labels)[
        probabilities.argmax(axis=1)
    ]

    score = f1_score(
        fold2_valid["label"].to_numpy(),
        predictions,
        labels=fold2_labels,
        average="macro",
        zero_division=0,
    )
    return score, probabilities


fold2_optimizer = torch.optim.AdamW(
    fold2_model.parameters(),
    lr=5e-6,
    weight_decay=0.01,
)

fold2_history = []
fold2_best_score = -1.0
fold2_best_epoch = None

for epoch in range(1, 3):
    fold2_model.train()
    loss_sum = 0.0
    sample_count = 0

    progress = tqdm(
        fold2_train_loader,
        desc=f"Fold 2 DistilRoBERTa epoch {epoch}/2",
    )

    for batch in progress:
        batch = {
            key: value.to(fold2_device)
            for key, value in batch.items()
        }

        fold2_optimizer.zero_grad(set_to_none=True)

        # 原版、不加權的交叉熵。
        loss = fold2_model(**batch).loss

        if not torch.isfinite(loss):
            raise RuntimeError("訓練 loss 非有限值，已停止。")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            fold2_model.parameters(), 1.0
        )
        fold2_optimizer.step()

        batch_size = batch["labels"].size(0)
        loss_sum += loss.item() * batch_size
        sample_count += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    score, probabilities = evaluate_fold2_distil()

    fold2_history.append({
        "epoch": epoch,
        "train_loss": loss_sum / sample_count,
        "validation_macro_f1": score,
    })

    print(
        f"Epoch {epoch}: "
        f"validation macro F1 = {score:.4f}"
    )

    if score > fold2_best_score:
        fold2_best_score = score
        fold2_best_epoch = epoch

        fold2_model.save_pretrained(fold2_dir)
        fold2_tokenizer.save_pretrained(fold2_dir)

        np.savez_compressed(
            fold2_dir / "validation_probabilities.npz",
            probabilities=probabilities,
            post_id=fold2_valid["post_id"].to_numpy(dtype=str),
            targets=fold2_valid["label"].to_numpy(dtype=str),
            labels=np.asarray(fold2_labels, dtype=str),
        )

pd.DataFrame(fold2_history).to_csv(
    fold2_dir / "training_history.csv",
    index=False,
)

# 儲存分組資料，後續讓 DeBERTa 使用完全相同的驗證集。
fold2_train[["post_id", "label"]].to_csv(
    fold2_dir / "train_rows.csv", index=False
)
fold2_valid[["post_id", "label"]].to_csv(
    fold2_dir / "validation_rows.csv", index=False
)

del fold2_optimizer
fold2_model.cpu()
gc.collect()
torch.cuda.empty_cache()

display(pd.DataFrame(fold2_history))
print("最佳 epoch：", fold2_best_epoch)
print("最佳 validation macro F1：", round(fold2_best_score, 4))
print("模型位置：", fold2_dir.resolve())

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Fold 2 DistilRoBERTa epoch 1/2:   0%|          | 0/337 [00:00<?, ?it/s]

Epoch 1: validation macro F1 = 0.6204


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fold 2 DistilRoBERTa epoch 2/2:   0%|          | 0/337 [00:00<?, ?it/s]

Epoch 2: validation macro F1 = 0.6269


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,epoch,train_loss,validation_macro_f1
0,1,1.251400,0.620432
1,2,1.011756,0.626933


最佳 epoch： 2
最佳 validation macro F1： 0.6269
模型位置： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\distilroberta_fold5_unweighted_20261125_222502_258593


In [61]:
import math
from transformers import get_linear_schedule_with_warmup

# 前一格的 fold2_train、fold2_valid、fold2_dir 必須仍存在。
fold2_distil_dir = Path(fold2_dir)

fold2_db_dir = Path("models") / (
    "deberta_fold5_unweighted_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
fold2_db_dir.mkdir(parents=True, exist_ok=False)

gc.collect()
torch.cuda.empty_cache()

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

fold2_db_tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/deberta-v3-base",
    use_fast=False,
)

fold2_db_label_to_id = {
    label: index
    for index, label in enumerate(fold2_labels)
}

fold2_db_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=len(fold2_labels),
    label2id=fold2_db_label_to_id,
    id2label={
        index: label
        for label, index in fold2_db_label_to_id.items()
    },
).to(device=fold2_device, dtype=torch.float32)
assert all(
    parameter.dtype == torch.float32
    for parameter in fold2_db_model.parameters()
    if parameter.requires_grad
)

print("模型參數 dtype：", next(fold2_db_model.parameters()).dtype)


class Fold2DebertaDataset(Dataset):
    def __init__(self, frame):
        self.encodings = fold2_db_tokenizer(
            frame["text"].fillna("").astype(str).tolist(),
            truncation=True,
            max_length=128,
        )
        self.targets = [
            fold2_db_label_to_id[label]
            for label in frame["label"]
        ]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        item = {
            key: values[index]
            for key, values in self.encodings.items()
        }
        item["labels"] = self.targets[index]
        return item


fold2_db_collator = DataCollatorWithPadding(
    tokenizer=fold2_db_tokenizer,
    pad_to_multiple_of=8,
)

fold2_db_train_loader = DataLoader(
    Fold2DebertaDataset(fold2_train),
    batch_size=4,
    shuffle=True,
    collate_fn=fold2_db_collator,
    num_workers=0,
)

fold2_db_valid_loader = DataLoader(
    Fold2DebertaDataset(fold2_valid),
    batch_size=4,
    shuffle=False,
    collate_fn=fold2_db_collator,
    num_workers=0,
)


@torch.inference_mode()
def evaluate_fold2_deberta():
    fold2_db_model.eval()
    batches = []

    for batch in fold2_db_valid_loader:
        inputs = {
            key: value.to(fold2_device)
            for key, value in batch.items()
            if key != "labels"
        }

        # 與先前驗證一致，推論使用 FP32。
        logits = fold2_db_model(**inputs).logits
        batches.append(
            torch.softmax(logits.float(), dim=1).cpu().numpy()
        )

    probabilities = np.concatenate(batches, axis=0)
    predictions = np.asarray(fold2_labels)[
        probabilities.argmax(axis=1)
    ]

    score = f1_score(
        fold2_valid["label"].to_numpy(),
        predictions,
        labels=fold2_labels,
        average="macro",
        zero_division=0,
    )
    return score, probabilities


fold2_db_epochs = 4
fold2_db_accum = 4

fold2_db_optimizer = torch.optim.AdamW(
    fold2_db_model.parameters(),
    lr=2e-5,
    weight_decay=0.01,
)

fold2_db_total_updates = (
    math.ceil(len(fold2_db_train_loader) / fold2_db_accum)
    * fold2_db_epochs
)

fold2_db_scheduler = get_linear_schedule_with_warmup(
    fold2_db_optimizer,
    num_warmup_steps=int(fold2_db_total_updates * 0.1),
    num_training_steps=fold2_db_total_updates,
)

fold2_db_scaler = torch.amp.GradScaler("cuda")
fold2_db_history = []
fold2_db_best_score = -1.0
fold2_db_best_epoch = None

for epoch in range(1, fold2_db_epochs + 1):
    fold2_db_model.train()
    fold2_db_optimizer.zero_grad(set_to_none=True)

    loss_sum = 0.0
    sample_count = 0

    progress = tqdm(
        fold2_db_train_loader,
        desc=f"Fold 2 DeBERTa epoch {epoch}/{fold2_db_epochs}",
    )

    for step, batch in enumerate(progress):
        batch = {
            key: value.to(fold2_device)
            for key, value in batch.items()
        }

        # 最後不足 4 批的累積組，使用實際批數。
        group_start = (
            step // fold2_db_accum
        ) * fold2_db_accum
        group_size = min(
            fold2_db_accum,
            len(fold2_db_train_loader) - group_start,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            raw_loss = fold2_db_model(**batch).loss
            loss = raw_loss / group_size

        if not torch.isfinite(raw_loss):
            raise RuntimeError("訓練 loss 非有限值，已停止。")

        fold2_db_scaler.scale(loss).backward()

        should_update = (
            (step + 1) % fold2_db_accum == 0
            or step + 1 == len(fold2_db_train_loader)
        )

        if should_update:
            fold2_db_scaler.unscale_(fold2_db_optimizer)
            torch.nn.utils.clip_grad_norm_(
                fold2_db_model.parameters(), 1.0
            )

            previous_scale = fold2_db_scaler.get_scale()
            fold2_db_scaler.step(fold2_db_optimizer)
            fold2_db_scaler.update()

            if fold2_db_scaler.get_scale() >= previous_scale:
                fold2_db_scheduler.step()

            fold2_db_optimizer.zero_grad(set_to_none=True)

        batch_size = batch["labels"].size(0)
        loss_sum += raw_loss.item() * batch_size
        sample_count += batch_size
        progress.set_postfix(loss=f"{raw_loss.item():.4f}")

    score, probabilities = evaluate_fold2_deberta()

    fold2_db_history.append({
        "epoch": epoch,
        "train_loss": loss_sum / sample_count,
        "validation_macro_f1": score,
    })

    print(
        f"Epoch {epoch}: validation macro F1 = {score:.4f}"
    )

    if score > fold2_db_best_score:
        fold2_db_best_score = score
        fold2_db_best_epoch = epoch

        fold2_db_model.save_pretrained(fold2_db_dir)
        fold2_db_tokenizer.save_pretrained(fold2_db_dir)

        np.savez_compressed(
            fold2_db_dir / "validation_probabilities.npz",
            probabilities=probabilities,
            post_id=fold2_valid["post_id"].to_numpy(dtype=str),
            targets=fold2_valid["label"].to_numpy(dtype=str),
            labels=np.asarray(fold2_labels, dtype=str),
        )

pd.DataFrame(fold2_db_history).to_csv(
    fold2_db_dir / "training_history.csv",
    index=False,
)

del fold2_db_optimizer, fold2_db_scheduler, fold2_db_scaler
fold2_db_model.cpu()
gc.collect()
torch.cuda.empty_cache()

display(pd.DataFrame(fold2_db_history))
print("最佳 epoch：", fold2_db_best_epoch)
print("最佳 validation macro F1：", round(fold2_db_best_score, 4))
print("模型位置：", fold2_db_dir.resolve())

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.den

模型參數 dtype： torch.float32


Fold 2 DeBERTa epoch 1/4:   0%|          | 0/673 [00:00<?, ?it/s]

Epoch 1: validation macro F1 = 0.2648


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (7 outputs left out)

Epoch 4: validation macro F1 = 0.6192


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,epoch,train_loss,validation_macro_f1
0,1,1.668917,0.264782
1,2,1.258716,0.531552
2,3,0.907132,0.590942
3,4,0.715655,0.619213


最佳 epoch： 4
最佳 validation macro F1： 0.6192
模型位置： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\deberta_fold5_unweighted_20261125_222537_959837


In [62]:
fold5_distil_dir = Path(fold2_distil_dir)
fold5_deberta_dir = Path(fold2_db_dir)

assert CURRENT_FOLD == 5
assert "distilroberta_fold5_" in fold5_distil_dir.name
assert "deberta_fold5_" in fold5_deberta_dir.name

# 確認驗證資料就是第 5 fold。
expected_valid = train.iloc[fold2_splits[4][1]]
assert np.array_equal(
    fold2_valid["post_id"].astype(str).to_numpy(),
    expected_valid["post_id"].astype(str).to_numpy(),
)

repair_valid = expected_valid.copy()
repair_labels = np.asarray(
    sorted(train["label"].unique().tolist()), dtype=str
)
repair_targets = repair_valid["label"].to_numpy(dtype=str)
repair_ids = repair_valid["post_id"].to_numpy(dtype=str)
repair_texts = (
    repair_valid["text"].fillna("").astype(str).tolist()
)

fold5_distil_probs = repair_predict(
    fold5_distil_dir, "roberta"
)
fold5_deberta_probs = repair_predict(
    fold5_deberta_dir, "deberta-v2"
)

assert not np.array_equal(
    fold5_distil_probs, fold5_deberta_probs
), "兩個模型的機率完全相同，請先檢查。"

fold5_results = []

for name, probabilities in {
    "Fold 5 DistilRoBERTa": fold5_distil_probs,
    "Fold 5 DeBERTa": fold5_deberta_probs,
    "Fold 5 Original 50/50": (
        0.5 * fold5_distil_probs
        + 0.5 * fold5_deberta_probs
    ),
}.items():
    fold5_results.append({
        "model": name,
        "validation_macro_f1": f1_score(
            repair_targets,
            repair_labels[probabilities.argmax(axis=1)],
            labels=repair_labels,
            average="macro",
            zero_division=0,
        ),
    })

for directory, probabilities in [
    (fold5_distil_dir, fold5_distil_probs),
    (fold5_deberta_dir, fold5_deberta_probs),
]:
    np.savez_compressed(
        directory / "validation_probabilities_repaired.npz",
        probabilities=probabilities,
        post_id=repair_ids,
        targets=repair_targets,
        labels=repair_labels,
    )

cv_model_dirs[5] = {
    "distil": fold5_distil_dir,
    "deberta": fold5_deberta_dir,
}

with open("cv_model_dirs.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            str(number): {
                name: str(path)
                for name, path in paths.items()
            }
            for number, paths in cv_model_dirs.items()
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

display(pd.DataFrame(fold5_results))
print("Fold 5 模型位置與驗證機率已保存。")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

重新推論 roberta:   0%|          | 0/168 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

重新推論 deberta-v2:   0%|          | 0/168 [00:00<?, ?it/s]

,model,validation_macro_f1
0,Fold 5 DistilRoBERTa,0.626933
1,Fold 5 DeBERTa,0.619213
2,Fold 5 Original 50/50,0.642836


Fold 5 模型位置與驗證機率已保存。


In [63]:
from datetime import datetime

## 最終提交一：五折 50/50 測試機率與 CSV Final Submission 1: 50/50 Test Split and CSV


In [64]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

# 所有輸入都從硬碟讀取，不依賴先前的變數。
root = Path.cwd()
backup_path = root / "fulltrain_original_predictions_backup.npz"
sample_path = root / "data/competition/sampleSubmission.csv"

assert backup_path.is_file(), f"找不到備份：{backup_path}"
assert sample_path.is_file(), f"找不到官方範例：{sample_path}"

with np.load(backup_path, allow_pickle=True) as saved:
    distil = saved["distil_probs"].astype(np.float64)
    deberta = saved["deberta_probs"].astype(np.float64)
    labels = saved["labels"].astype(str)
    test_ids = saved["post_id"].astype(str)

sample = pd.read_csv(sample_path, dtype=str)
id_column = "post_id" if "post_id" in sample.columns else "id"
submission_ids = sample[id_column].astype(str)

# 檢查備份是否完整。
assert labels.ndim == 1 and len(labels) == 6, "備份類別數不正確"
assert len(set(labels)) == 6, "備份類別重複"
assert test_ids.ndim == 1, "備份 ID 格式不正確"
assert len(set(test_ids)) == len(test_ids), "備份 ID 重複"
assert submission_ids.is_unique, "官方範例 ID 重複"
assert set(test_ids) == set(submission_ids), "備份與官方範例 ID 不一致"

expected_shape = (len(test_ids), len(labels))
for name, probs in [("DistilRoBERTa", distil), ("DeBERTa", deberta)]:
    assert probs.shape == expected_shape, f"{name} 機率維度不正確"
    assert np.isfinite(probs).all(), f"{name} 有無效數值"
    assert (probs >= 0).all(), f"{name} 有負機率"
    assert np.allclose(
        probs.sum(axis=1), 1.0, atol=1e-3
    ), f"{name} 每列機率總和不為 1"

# 依備份中的類別順序融合，再按官方 ID 順序排列。
ensemble_probs = 0.5 * distil + 0.5 * deberta
predictions = labels[ensemble_probs.argmax(axis=1)]

prediction_map = pd.Series(predictions, index=test_ids)
submission = pd.DataFrame({
    "id": submission_ids.to_numpy(),
    "label": submission_ids.map(prediction_map).to_numpy(),
})

assert submission.notna().all().all(), "提交檔有缺漏"
assert len(submission) == len(sample), "提交筆數不一致"

# 使用新目錄，保留原本的提交檔。
output_dir = root / "models" / (
    "restored_fulltrain_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
output_dir.mkdir(parents=True, exist_ok=False)
output_path = output_dir / "submission_fulltrain_distil_deberta_50_50.csv"
submission.to_csv(output_path, index=False)

print("完成：已讀回備份並產生全資料 50/50 提交檔，未重新訓練。")
print("提交筆數：", len(submission))
print("CSV 位置：", output_path.resolve())
print(submission.head().to_string(index=False))

完成：已讀回備份並產生全資料 50/50 提交檔，未重新訓練。
提交筆數： 1440
CSV 位置： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\restored_fulltrain_20261125_223220_690626\submission_fulltrain_distil_deberta_50_50.csv
     id    label
xyzx9jp surprise
l3vzzxt    anger
g680iug  sadness
m4t8bp6      joy
ai3al6e surprise


In [65]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)
from tqdm.auto import tqdm

assert torch.cuda.is_available()

# 從已保存的登記檔讀取五折模型位置。
with open("cv_model_dirs.json", encoding="utf-8") as file:
    cv_paths = {
        int(number): {
            name: Path(path)
            for name, path in paths.items()
        }
        for number, paths in json.load(file).items()
    }

assert set(cv_paths) == {1, 2, 3, 4, 5}, (
    f"模型登記不完整：{sorted(cv_paths)}"
)

all_paths = [
    cv_paths[fold][kind].resolve()
    for fold in range(1, 6)
    for kind in ("distil", "deberta")
]
assert len(set(all_paths)) == 10, "有模型路徑重複，請先檢查。"

for path in all_paths:
    assert (path / "config.json").exists(), str(path)

# 清理可能仍占用 GPU 的舊模型。
for name in (
    "ft_model", "deberta_model",
    "fold2_model", "fold2_db_model",
):
    old_model = globals().get(name)
    if isinstance(old_model, torch.nn.Module):
        old_model.cpu()

gc.collect()
torch.cuda.empty_cache()

cv_device = torch.device("cuda")
cv_labels = np.asarray(
    sorted(train["label"].unique().tolist()), dtype=str
)
cv_test_ids = test["post_id"].to_numpy(dtype=str)
cv_test_texts = test["text"].fillna("").astype(str).tolist()

assert test["post_id"].astype(str).is_unique

cv_output_dir = Path("models") / (
    "fivefold_predictions_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
cv_output_dir.mkdir(parents=True, exist_ok=False)


@torch.inference_mode()
def predict_cv_test(model_dir, expected_type, description):
    tokenizer = AutoTokenizer.from_pretrained(
        model_dir,
        use_fast=False,
        local_files_only=True,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_dir,
        local_files_only=True,
    )

    assert model.config.model_type == expected_type, (
        f"模型類型不符：{model_dir}"
    )

    model = model.to(cv_device).eval()

    try:
        label_to_id = {
            str(label).lower(): int(index)
            for index, label in model.config.id2label.items()
        }
        selected_ids = [
            label_to_id[label] for label in cv_labels
        ]

        batches = []
        for start in tqdm(
            range(0, len(cv_test_texts), 4),
            desc=description,
        ):
            inputs = tokenizer(
                cv_test_texts[start:start + 4],
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt",
            ).to(cv_device)

            logits = model(**inputs).logits
            probabilities = torch.softmax(
                logits[:, selected_ids].float(), dim=1
            )
            batches.append(probabilities.cpu().numpy())

        result = np.concatenate(batches, axis=0)

        assert result.shape == (len(test), len(cv_labels))
        assert np.isfinite(result).all()
        assert np.allclose(result.sum(axis=1), 1, atol=1e-5)
        return result

    finally:
        model.cpu()
        del model
        gc.collect()
        torch.cuda.empty_cache()


cv_test_predictions = {"distil": [], "deberta": []}

for fold in range(1, 6):
    for kind, model_type in [
        ("distil", "roberta"),
        ("deberta", "deberta-v2"),
    ]:
        probabilities = predict_cv_test(
            cv_paths[fold][kind],
            model_type,
            f"Fold {fold} {kind} 測試集",
        )

        cv_test_predictions[kind].append(probabilities)

        # 每個模型完成就保存，方便後續重用。
        np.savez_compressed(
            cv_output_dir / f"fold{fold}_{kind}_test.npz",
            probabilities=probabilities,
            post_id=cv_test_ids,
            labels=cv_labels,
        )

cv_distil_mean = np.mean(
    cv_test_predictions["distil"], axis=0
)
cv_deberta_mean = np.mean(
    cv_test_predictions["deberta"], axis=0
)

cv_ensemble_probs = (
    0.5 * cv_distil_mean
    + 0.5 * cv_deberta_mean
)

np.savez_compressed(
    cv_output_dir / "fivefold_ensemble_test.npz",
    probabilities=cv_ensemble_probs,
    distil_probabilities=cv_distil_mean,
    deberta_probabilities=cv_deberta_mean,
    post_id=cv_test_ids,
    labels=cv_labels,
)

# 按官方 sample_submission 的 ID 順序輸出。
sample_id_column = (
    "post_id"
    if "post_id" in sample_submission.columns
    else "id"
)
cv_submission_ids = sample_submission[
    sample_id_column
].astype(str)

assert cv_submission_ids.is_unique
assert len(cv_submission_ids) == len(test)
assert set(cv_submission_ids) == set(cv_test_ids)

cv_prediction_map = pd.Series(
    cv_labels[cv_ensemble_probs.argmax(axis=1)],
    index=cv_test_ids,
)

submission_fivefold = pd.DataFrame({
    "id": cv_submission_ids.to_numpy(),
    "label": cv_submission_ids.map(
        cv_prediction_map
    ).to_numpy(),
})

assert submission_fivefold.notna().all().all()
assert submission_fivefold["id"].is_unique
assert set(submission_fivefold["label"]).issubset(
    set(cv_labels)
)

cv_csv_path = (
    cv_output_dir
    / "submission_fivefold_distil_deberta_50_50.csv"
)
submission_fivefold.to_csv(cv_csv_path, index=False)

print("提交筆數：", len(submission_fivefold))
print("CSV 位置：", cv_csv_path.resolve())
display(submission_fivefold.head())

# 與目前最佳提交比較，僅檢查預測差異。
best_csv_path = Path(
    "submission_fulltrain_distil_deberta_50_50.csv"
)

if best_csv_path.exists():
    best_submission = pd.read_csv(
        best_csv_path, dtype=str
    ).set_index("id")

    assert best_submission.index.is_unique
    assert set(submission_fivefold["id"]).issubset(
        set(best_submission.index)
    )

    old_labels = best_submission.loc[
        submission_fivefold["id"], "label"
    ].to_numpy()

    changed = (
        old_labels
        != submission_fivefold["label"].to_numpy()
    )

    print("與目前最佳提交不同筆數：", int(changed.sum()))
    print("不同預測比例：", f"{changed.mean():.2%}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Fold 1 distil 測試集:   0%|          | 0/360 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Fold 1 deberta 測試集:   0%|          | 0/360 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Fold 2 distil 測試集:   0%|          | 0/360 [00:00<?, ?it/s]

... (12 outputs left out)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Fold 5 deberta 測試集:   0%|          | 0/360 [00:00<?, ?it/s]

提交筆數： 1440
CSV 位置： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\fivefold_predictions_20261125_223220_896893\submission_fivefold_distil_deberta_50_50.csv


,id,label
0,xyzx9jp,surprise
1,l3vzzxt,anger
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise


## 補充分析：5 fold OOF 與融合比例  Supplementary Analysis: 5-Fold OOF and Blending Ratios


In [66]:
from sklearn.metrics import f1_score

# 檢查五折驗證資料恰好涵蓋全部訓練列一次。
oof_coverage = np.zeros(len(train), dtype=int)

oof_distil = np.zeros(
    (len(train), len(cv_labels)), dtype=np.float32
)
oof_deberta = np.zeros_like(oof_distil)

# 暫存 repair_predict 使用的全域變數。
repair_backup = (
    repair_texts, repair_labels, repair_targets, repair_ids
)

try:
    for fold in range(1, 6):
        valid_idx = fold2_splits[fold - 1][1]
        valid_frame = train.iloc[valid_idx]

        repair_labels = cv_labels.copy()
        repair_targets = valid_frame["label"].to_numpy(dtype=str)
        repair_ids = valid_frame["post_id"].to_numpy(dtype=str)
        repair_texts = (
            valid_frame["text"].fillna("").astype(str).tolist()
        )

        distil_probs = repair_predict(
            cv_paths[fold]["distil"], "roberta"
        )
        deberta_probs = repair_predict(
            cv_paths[fold]["deberta"], "deberta-v2"
        )

        oof_distil[valid_idx] = distil_probs
        oof_deberta[valid_idx] = deberta_probs
        oof_coverage[valid_idx] += 1

finally:
    (
        repair_texts, repair_labels,
        repair_targets, repair_ids
    ) = repair_backup

assert np.all(oof_coverage == 1)
assert np.isfinite(oof_distil).all()
assert np.isfinite(oof_deberta).all()

oof_targets = train["label"].to_numpy(dtype=str)

np.savez_compressed(
    cv_output_dir / "fivefold_oof_verified.npz",
    distil_probabilities=oof_distil,
    deberta_probabilities=oof_deberta,
    post_id=train["post_id"].to_numpy(dtype=str),
    targets=oof_targets,
    labels=cv_labels,
)

weight_results = []

for weight in [0.50, 0.65, 0.75, 1.00]:
    probabilities = (
        weight * oof_distil
        + (1 - weight) * oof_deberta
    )
    predictions = cv_labels[probabilities.argmax(axis=1)]

    row = {
        "Distil_weight": weight,
        "DeBERTa_weight": 1 - weight,
        "OOF_macro_f1": f1_score(
            oof_targets,
            predictions,
            labels=cv_labels,
            average="macro",
            zero_division=0,
        ),
    }

    fold_scores = []
    for fold in range(1, 6):
        valid_idx = fold2_splits[fold - 1][1]

        score = f1_score(
            oof_targets[valid_idx],
            predictions[valid_idx],
            labels=cv_labels,
            average="macro",
            zero_division=0,
        )
        row[f"fold{fold}"] = score
        fold_scores.append(score)

    row["fold_mean"] = np.mean(fold_scores)
    weight_results.append(row)

weight_table = pd.DataFrame(weight_results).sort_values(
    "OOF_macro_f1", ascending=False
).reset_index(drop=True)

display(weight_table.round(6))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

重新推論 roberta:   0%|          | 0/169 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

重新推論 deberta-v2:   0%|          | 0/169 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

重新推論 roberta:   0%|          | 0/168 [00:00<?, ?it/s]

... (11 outputs left out)

重新推論 roberta:   0%|          | 0/168 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

重新推論 deberta-v2:   0%|          | 0/168 [00:00<?, ?it/s]

,Distil_weight,DeBERTa_weight,OOF_macro_f1,fold1,fold2,fold3,fold4,fold5,fold_mean
0,0.75,0.25,0.631680,0.605943,0.649012,0.616791,0.643583,0.643042,0.631674
1,0.65,0.35,0.629801,0.597004,0.652687,0.622462,0.633679,0.643336,0.629834
2,0.50,0.50,0.627617,0.596547,0.640692,0.636828,0.619718,0.642836,0.627324
3,1.00,0.00,0.621424,0.591306,0.638419,0.615704,0.632793,0.626933,0.621031


### 幾何平均驗證比較 Comparison of Geometric Mean Verification


In [67]:
# 比較目前算術平均與幾何平均。
# oof_distil、oof_deberta 是剛才重新推論的五折驗證機率。

eps = 1e-12

arithmetic_oof = 0.5 * oof_distil + 0.5 * oof_deberta

geometric_oof = np.sqrt(
    np.clip(oof_distil, eps, 1)
    * np.clip(oof_deberta, eps, 1)
)
geometric_oof /= geometric_oof.sum(axis=1, keepdims=True)

fusion_results = []

for name, probabilities in {
    "Arithmetic 50/50": arithmetic_oof,
    "Geometric 50/50": geometric_oof,
}.items():
    predictions = cv_labels[probabilities.argmax(axis=1)]

    row = {
        "method": name,
        "OOF_macro_f1": f1_score(
            oof_targets,
            predictions,
            labels=cv_labels,
            average="macro",
            zero_division=0,
        ),
    }

    for fold in range(1, 6):
        idx = fold2_splits[fold - 1][1]

        row[f"fold{fold}"] = f1_score(
            oof_targets[idx],
            predictions[idx],
            labels=cv_labels,
            average="macro",
            zero_division=0,
        )

    fusion_results.append(row)

display(pd.DataFrame(fusion_results).round(6))

,method,OOF_macro_f1,fold1,fold2,fold3,fold4,fold5
0,Arithmetic 50/50,0.627617,0.596547,0.640692,0.636828,0.619718,0.642836
1,Geometric 50/50,0.626155,0.593776,0.633562,0.642745,0.614549,0.644476


### 幾何平均候選 CSV（非最終選取）Geometric Mean Candidate CSV (Not Final Selection)


In [68]:
# 先對每種架構取五折平均，再做兩架構的幾何平均。
eps = 1e-12

final_geo_probs = np.sqrt(
    np.clip(cv_distil_mean, eps, 1)
    * np.clip(cv_deberta_mean, eps, 1)
)
final_geo_probs /= final_geo_probs.sum(axis=1, keepdims=True)

assert np.isfinite(final_geo_probs).all()

final_geo_predictions = cv_labels[
    final_geo_probs.argmax(axis=1)
]

original_predictions = cv_labels[
    (0.5 * cv_distil_mean + 0.5 * cv_deberta_mean)
    .argmax(axis=1)
]

changed = int(np.sum(
    final_geo_predictions != original_predictions
))
print("與 0.6458 版本不同筆數：", changed)

geo_map = pd.Series(
    final_geo_predictions,
    index=cv_test_ids,
)

final_geo_submission = pd.DataFrame({
    "id": cv_submission_ids.to_numpy(),
    "label": cv_submission_ids.map(geo_map).to_numpy(),
})

assert len(final_geo_submission) == len(test)
assert final_geo_submission["id"].is_unique
assert final_geo_submission.notna().all().all()
assert set(final_geo_submission["label"]).issubset(set(cv_labels))

geo_path = (
    cv_output_dir
    / "submission_fivefold_geometric_50_50.csv"
)

if changed == 0:
    print("預測完全相同，不用提交。")
else:
    final_geo_submission.to_csv(geo_path, index=False)
    print("候選檔案：", geo_path.resolve())
    display(final_geo_submission.head())

與 0.6458 版本不同筆數： 43
候選檔案： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\fivefold_predictions_20261125_223220_896893\submission_fivefold_geometric_50_50.csv


,id,label
0,xyzx9jp,surprise
1,l3vzzxt,anger
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise


### 溫度設定比較（非完整 nested CV）Temperature Setting Comparison (Incomplete Nested CV)


In [69]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

def temperature_probs(probabilities, temperature):
    log_probs = np.log(np.clip(probabilities, 1e-12, 1))
    scaled = log_probs / temperature
    scaled -= scaled.max(axis=1, keepdims=True)
    result = np.exp(scaled)
    return result / result.sum(axis=1, keepdims=True)

def macro_score(targets, probabilities):
    return f1_score(
        targets,
        cv_labels[probabilities.argmax(axis=1)],
        labels=cv_labels,
        average="macro",
        zero_division=0,
    )

# T > 1：機率更平緩；T < 1：機率更尖銳。
# 保留原版 (1, 1)，只測小範圍。
temperature_candidates = [
    (1.0, 1.0),
    (1.0, 1.5),
    (1.0, 2.0),
    (1.5, 1.0),
    (2.0, 1.0),
    (1.5, 1.5),
    (0.75, 0.75),
]

temperature_oof = {}
temperature_rows = []

for distil_t, deberta_t in temperature_candidates:
    probs = (
        0.5 * temperature_probs(oof_distil, distil_t)
        + 0.5 * temperature_probs(oof_deberta, deberta_t)
    )
    temperature_oof[(distil_t, deberta_t)] = probs

    row = {
        "Distil_T": distil_t,
        "DeBERTa_T": deberta_t,
        "OOF_macro_f1": macro_score(oof_targets, probs),
    }

    for fold in range(1, 6):
        idx = fold2_splits[fold - 1][1]
        row[f"fold{fold}"] = macro_score(
            oof_targets[idx], probs[idx]
        )

    temperature_rows.append(row)

temperature_table = pd.DataFrame(
    temperature_rows
).sort_values(
    "OOF_macro_f1", ascending=False
)

print("各設定的直接 OOF 比較：")
display(temperature_table.round(6))

# 用其餘四折挑溫度，在留下的一折檢查。
temperature_check_rows = []
temperature_selected_oof = np.zeros_like(oof_distil)
temperature_coverage = np.zeros(len(train), dtype=int)

for fold in range(1, 6):
    held_idx = fold2_splits[fold - 1][1]
    selection_mask = np.ones(len(train), dtype=bool)
    selection_mask[held_idx] = False

    # 分數相同時優先原版，因為它排在候選清單第一個。
    selected = max(
        temperature_candidates,
        key=lambda setting: macro_score(
            oof_targets[selection_mask],
            temperature_oof[setting][selection_mask],
        ),
    )

    selected_probs = temperature_oof[selected][held_idx]
    baseline_probs = temperature_oof[(1.0, 1.0)][held_idx]

    selected_score = macro_score(
        oof_targets[held_idx], selected_probs
    )
    baseline_score = macro_score(
        oof_targets[held_idx], baseline_probs
    )

    temperature_selected_oof[held_idx] = selected_probs
    temperature_coverage[held_idx] += 1

    temperature_check_rows.append({
        "held_fold": fold,
        "selected_Distil_T": selected[0],
        "selected_DeBERTa_T": selected[1],
        "baseline_f1": baseline_score,
        "selected_f1": selected_score,
        "difference": selected_score - baseline_score,
    })

assert np.all(temperature_coverage == 1)

print("四折選設定、另一折檢查：")
display(pd.DataFrame(temperature_check_rows).round(6))

print(
    "原版整體 OOF：",
    round(macro_score(
        oof_targets, temperature_oof[(1.0, 1.0)]
    ), 6),
)
print(
    "各折選設定後整體 OOF：",
    round(macro_score(
        oof_targets, temperature_selected_oof
    ), 6),
)

各設定的直接 OOF 比較：


,Distil_T,DeBERTa_T,OOF_macro_f1,fold1,fold2,fold3,fold4,fold5
2,1.00,2.00,0.631413,0.597857,0.651736,0.628604,0.637546,0.641298
1,1.00,1.50,0.630255,0.598595,0.646700,0.627310,0.634353,0.643955
0,1.00,1.00,0.627617,0.596547,0.640692,0.636828,0.619718,0.642836
5,1.50,1.50,0.627400,0.594229,0.637458,0.638610,0.617713,0.646452
6,0.75,0.75,0.626557,0.596011,0.638918,0.636715,0.617856,0.641768
3,1.50,1.00,0.616689,0.589105,0.624877,0.628742,0.598420,0.638804
4,2.00,1.00,0.615980,0.579401,0.623758,0.634488,0.596793,0.641366


四折選設定、另一折檢查：


,held_fold,selected_Distil_T,selected_DeBERTa_T,baseline_f1,selected_f1,difference
0,1,1.0,2.0,0.596547,0.597857,0.001310
1,2,1.0,2.0,0.640692,0.651736,0.011045
2,3,1.0,2.0,0.636828,0.628604,-0.008223
3,4,1.0,2.0,0.619718,0.637546,0.017828
4,5,1.0,2.0,0.642836,0.641298,-0.001538


原版整體 OOF： 0.627617
各折選設定後整體 OOF： 0.631413


### 溫度調整候選 CSV（未選取）Temperature Adjustment Candidates CSV (Not Selected)


In [70]:
# 與 OOF 實驗一致：
# 每個 fold 先調整溫度，再將五折機率平均。
# 不要先平均五折再調溫度，兩者結果不同。

temp_distil_mean = np.mean(
    [
        temperature_probs(probabilities, 1.0)
        for probabilities in cv_test_predictions["distil"]
    ],
    axis=0,
)

temp_deberta_mean = np.mean(
    [
        temperature_probs(probabilities, 1.5)
        for probabilities in cv_test_predictions["deberta"]
    ],
    axis=0,
)

temp_test_probs = (
    0.5 * temp_distil_mean
    + 0.5 * temp_deberta_mean
)

assert np.isfinite(temp_test_probs).all()
assert np.allclose(temp_test_probs.sum(axis=1), 1, atol=1e-5)

temp_predictions = cv_labels[
    temp_test_probs.argmax(axis=1)
]

baseline_predictions = cv_labels[
    (0.5 * cv_distil_mean + 0.5 * cv_deberta_mean)
    .argmax(axis=1)
]

changed = int(np.sum(
    temp_predictions != baseline_predictions
))
print("與 0.6458 版本不同筆數：", changed)

temp_prediction_map = pd.Series(
    temp_predictions,
    index=cv_test_ids,
)

temp_submission = pd.DataFrame({
    "id": cv_submission_ids.to_numpy(),
    "label": cv_submission_ids.map(temp_prediction_map).to_numpy(),
})

assert len(temp_submission) == len(test)
assert temp_submission["id"].is_unique
assert temp_submission.notna().all().all()
assert set(temp_submission["label"]).issubset(set(cv_labels))

temp_path = (
    cv_output_dir
    / "submission_fivefold_50_50_temperature_1_1p5.csv"
)

if changed == 0:
    print("預測與原版完全相同，不需要提交。")
else:
    temp_submission.to_csv(temp_path, index=False)

    np.savez_compressed(
        cv_output_dir / "temperature_1_1p5_test.npz",
        probabilities=temp_test_probs,
        post_id=cv_test_ids,
        labels=cv_labels,
    )

    print("候選 CSV：", temp_path.resolve())
    display(temp_submission.head())

與 0.6458 版本不同筆數： 54
候選 CSV： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\models\fivefold_predictions_20261125_223220_896893\submission_fivefold_50_50_temperature_1_1p5.csv


,id,label
0,xyzx9jp,surprise
1,l3vzzxt,anger
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise


## Baseline Submission（歷史 SVM 版本）(Historical SVM Versions)

最終選取為五折 50/50 與全資料 50/50；下格僅重現早期 SVM baseline。

The final selections are the 50/50 split (5-fold) and the 50/50 split using the full dataset; the bottom section merely reproduces the early SVM baseline.



In [71]:
from sklearn.base import clone

final_word_char_model = clone(tuned_word_char_model)
final_word_char_model.fit(train["clean"], train["label"])

predictions = final_word_char_model.predict(test["clean"])

assert test["post_id"].is_unique

prediction_map = pd.Series(
    predictions,
    index=test["post_id"],
)

submission_word_char = pd.DataFrame({
    "id": sample_submission["post_id"],
    "label": sample_submission["post_id"].map(prediction_map),
})

assert submission_word_char.notna().all().all()
assert submission_word_char["id"].is_unique
assert len(submission_word_char) == len(test)
assert set(submission_word_char["label"]).issubset(
    set(train["label"])
)

output_path = Path("submission_word_char.csv")
submission_word_char.to_csv(output_path, index=False)

print("提交筆數：", len(submission_word_char))
print("儲存位置：", output_path.resolve())
display(submission_word_char.head())

提交筆數： 1440
儲存位置： C:\Users\USER\Documents\Data Mining\lab2\DM2026-Lab2-Exercise\submission_word_char.csv


,id,label
0,xyzx9jp,anger
1,l3vzzxt,fear
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise
